# decision tree

In [ ]:
"""
Decision Tree (Binary Classification) on UNSW-NB15 -- FULL FEATURE SET
========================================================================
Goal: Train a hyperparameter-tuned Decision Tree classifier using ALL
features of the UNSW-NB15 dataset (NO feature selection / dimensionality
reduction of any kind) and benchmark it against the Decision Tree
numbers reported in three reference papers:

  [1] Alkhater, N. (2026). "A Rigorous Comparative Study of Supervised
      Machine Learning Techniques for Network Anomaly Detection:
      Empirical Insights from the UNSW-NB15 Dataset." Computers, 15(5),
      285.
      -> DT (max_depth=10, 5-fold stratified CV, 48 features):
         Acc=0.93+-0.012, Prec=0.92+-0.015, Recall=0.91+-0.011,
         F1=0.91+-0.013, AUC=0.940+-0.008

  [2] Kasongo, S.M. & Sun, Y. (2020). "Performance Analysis of Intrusion
      Detection Systems Using a Feature Selection Method on the
      UNSW-NB15 Dataset." Journal of Big Data, 7:105.
      -> DT, FULL 42-feature space (no XGBoost feature selection),
         binary classification:
         Test Acc=88.13%, Precision=83.91%, Recall=96.47%, F1=90.00%

  [3] Mohale, V.Z. & Obagbuwa, I.C. (2025). "Evaluating machine
      learning-based intrusion detection systems with explainable AI:
      enhancing transparency and interpretability." Frontiers in
      Computer Science, 7:1520741.
      -> DT: Acc=87%, Precision=0.85, Recall=0.88, F1=0.86,
         ROC-AUC=0.92, FPR=0.07, FNR=0.12

IMPORTANT
---------
This script performs NO feature selection step. Every original column
(after only the mandatory cleaning / categorical-encoding steps that
each of the three papers also performs -- dropping the row-id column
and encoding proto/service/state) is kept and fed to the model.

HOW TO USE
----------
1. Download the UNSW-NB15 dataset yourself, e.g. the commonly used
   header files:
       UNSW_NB15_training-set.csv
       UNSW_NB15_testing-set.csv
   (official source: UNSW Canberra Cyber / ADFA-NB15 repository,
    also mirrored on Kaggle).
2. Put them in the same folder as this script (or edit the paths in
   the __main__ block below).
3. Run:  python decision_tree_unsw_nb15_full_features.py
4. Check the ./outputs/ folder for:
     - comparison_table.csv
     - comparison_bar.png
     - confusion_matrix.png
     - roc_curve.png
     - feature_importance.png
     - tree_structure.png
"""

import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
)
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)


# ---------------------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------------------
def load_unsw_nb15(train_path=None, test_path=None, single_path=None):
    """
    Flexible loader.
    - train_path & test_path -> standard 'UNSW_NB15_training-set.csv' /
      'UNSW_NB15_testing-set.csv' files (each already has 'attack_cat'
      and 'label' columns). They are concatenated so the WHOLE dataset
      (~257k records, matching the scale used in the reference papers)
      goes through the same 5-fold CV protocol.
    - single_path -> one combined CSV containing a binary target column
      named 'label' or 'Label'.
    """
    if train_path and test_path:
        df_train = pd.read_csv("/kaggle/input/datasets/ajeetkumar20/unsw-nb15-v1-1/UNSW_NB15_training-set.csv")
        df_test = pd.read_csv("/kaggle/input/datasets/ajeetkumar20/unsw-nb15-v1-1/UNSW_NB15_testing-set.csv")
        df = pd.concat([df_train, df_test], axis=0, ignore_index=True)
    elif single_path:
        df = pd.read_csv(single_path)
    else:
        raise ValueError("Provide either (train_path & test_path) or single_path")
    return df


# ---------------------------------------------------------------------
# 2. PREPROCESSING  (NO FEATURE SELECTION -- every column kept)
# ---------------------------------------------------------------------
def preprocess(df):
    df = df.copy()

    # Drop only the row-identifier column (not a real feature -- every
    # one of the 3 reference papers drops it too before modelling)
    for id_col in ["id", "ID", "Id"]:
        if id_col in df.columns:
            df = df.drop(columns=[id_col])

    # Identify the binary target
    if "label" in df.columns:
        y = df["label"].astype(int)
        df = df.drop(columns=["label"])
    elif "Label" in df.columns:
        y = df["Label"].astype(int)
        df = df.drop(columns=["Label"])
    else:
        raise ValueError("Could not find a binary 'label' column in the data.")

    # attack_cat directly names the attack type -> perfectly leaks the
    # binary label. Every reference paper excludes it from the binary
    # feature set too, so we drop it here (this is NOT feature
    # selection on the traffic features -- it's removing a label leak).
    if "attack_cat" in df.columns:
        df = df.drop(columns=["attack_cat"])

    # Encode the categorical columns (proto, service, state) exactly as
    # all three papers do -- Label Encoding, so column COUNT is
    # unchanged (unlike one-hot, nothing is expanded or dropped).
    cat_cols = [c for c in ["proto", "service", "state"] if c in df.columns]
    for c in cat_cols:
        df[c] = LabelEncoder().fit_transform(df[c].astype(str))

    # Any other stray non-numeric columns -> label-encode too, so that
    # literally ALL remaining columns are usable model inputs
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = LabelEncoder().fit_transform(df[c].astype(str))

    # Min-Max scale to [0,1] (matches the papers' preprocessing pipeline)
    scaler = MinMaxScaler()
    X = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

    print(f"Final feature matrix: {X.shape[1]} features "
          f"(ALL features retained -- no selection/reduction applied)")
    return X, y


# ---------------------------------------------------------------------
# 3. HYPERPARAMETER TUNING (2-stage: RandomizedSearch -> GridSearch)
# ---------------------------------------------------------------------
def tune_decision_tree(X_train, y_train):
    base_dt = DecisionTreeClassifier(random_state=RANDOM_STATE)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    # Stage 1 -- broad RandomizedSearchCV to explore the space cheaply
    param_dist = {
        "criterion": ["gini", "entropy", "log_loss"],
        "max_depth": [6, 8, 10, 12, 14, 16, 18, 20, 25, None],
        "min_samples_split": [2, 4, 6, 10, 15, 20, 30],
        "min_samples_leaf": [1, 2, 4, 6, 10, 15],
        "max_features": [None, "sqrt", "log2"],
        "class_weight": [None, "balanced"],
        "ccp_alpha": [0.0, 1e-5, 1e-4, 5e-4, 1e-3],
    }

    rand_search = RandomizedSearchCV(
        base_dt, param_distributions=param_dist, n_iter=80,
        scoring="f1", cv=cv, n_jobs=-1, random_state=RANDOM_STATE, verbose=1
    )
    rand_search.fit(X_train, y_train)
    best_rand = rand_search.best_params_
    print("Stage-1 (Randomized) best params:", best_rand)

    # Stage 2 -- narrow GridSearchCV around the best region found above
    depth_center = best_rand["max_depth"]
    depth_options = {None}
    if depth_center is not None:
        depth_options |= {max(2, depth_center - 2), depth_center, depth_center + 2}

    grid = {
        "criterion": [best_rand["criterion"]],
        "max_depth": sorted(depth_options, key=lambda v: (v is None, v)),
        "min_samples_split": sorted({
            max(2, best_rand["min_samples_split"] - 2),
            best_rand["min_samples_split"],
            best_rand["min_samples_split"] + 2,
        }),
        "min_samples_leaf": sorted({
            max(1, best_rand["min_samples_leaf"] - 1),
            best_rand["min_samples_leaf"],
            best_rand["min_samples_leaf"] + 1,
        }),
        "max_features": [best_rand["max_features"]],
        "class_weight": [best_rand["class_weight"]],
        "ccp_alpha": sorted({0.0, best_rand["ccp_alpha"]}),
    }

    grid_search = GridSearchCV(
        base_dt, param_grid=grid, scoring="f1", cv=cv, n_jobs=-1, verbose=1
    )
    grid_search.fit(X_train, y_train)
    print("Stage-2 (Grid) best params:", grid_search.best_params_)
    print("Best CV F1:", grid_search.best_score_)

    return grid_search.best_estimator_, grid_search.best_params_


# ---------------------------------------------------------------------
# 4. EVALUATION (5-fold stratified CV, same protocol as [1])
# ---------------------------------------------------------------------
def evaluate(model, X, y, cv_splits=5):
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
    accs, precs, recs, f1s, aucs = [], [], [], [], []

    for train_idx, test_idx in cv.split(X, y):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        y_proba = model.predict_proba(X_te)[:, 1]

        accs.append(accuracy_score(y_te, y_pred))
        precs.append(precision_score(y_te, y_pred))
        recs.append(recall_score(y_te, y_pred))
        f1s.append(f1_score(y_te, y_pred))
        aucs.append(roc_auc_score(y_te, y_proba))

    return {
        "Accuracy": (np.mean(accs), np.std(accs)),
        "Precision": (np.mean(precs), np.std(precs)),
        "Recall": (np.mean(recs), np.std(recs)),
        "F1-Score": (np.mean(f1s), np.std(f1s)),
        "AUC": (np.mean(aucs), np.std(aucs)),
    }


# ---------------------------------------------------------------------
# 5. COMPARISON TABLE (against the 3 papers' published DT numbers)
# ---------------------------------------------------------------------
REFERENCE_RESULTS = {
    "Alkhater (2026) - DT": {
        "Accuracy": 0.930, "Precision": 0.920, "Recall": 0.910,
        "F1-Score": 0.910, "AUC": 0.940,
    },
    "Kasongo & Sun (2020) - DT (42 features, full space)": {
        "Accuracy": 0.8813, "Precision": 0.8391, "Recall": 0.9647,
        "F1-Score": 0.9000, "AUC": np.nan,
    },
    "Mohale & Obagbuwa (2025) - DT": {
        "Accuracy": 0.870, "Precision": 0.850, "Recall": 0.880,
        "F1-Score": 0.860, "AUC": 0.920,
    },
}


def build_comparison_table(my_results, out_csv=os.path.join(OUT_DIR, "comparison_table.csv")):
    rows = []
    for name, metrics in REFERENCE_RESULTS.items():
        row = {"Model": name}
        row.update(metrics)
        rows.append(row)

    my_row = {"Model": "Proposed Tuned DT (ALL features, no selection)"}
    for k, (mean, _std) in my_results.items():
        my_row[k] = round(mean, 4)
    rows.append(my_row)

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(df.to_string(index=False))
    return df


# ---------------------------------------------------------------------
# 6. GRAPHS
# ---------------------------------------------------------------------
def plot_comparison_bar(df, out_path=os.path.join(OUT_DIR, "comparison_bar.png")):
    metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]
    x = np.arange(len(df))
    width = 0.2
    fig, ax = plt.subplots(figsize=(11, 6))
    for i, m in enumerate(metrics):
        ax.bar(x + i * width, df[m], width, label=m)
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(df["Model"], rotation=20, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title("Decision Tree -- Proposed Model vs. Reference Papers")
    ax.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_confusion_matrix(model, X_test, y_test, out_path=os.path.join(OUT_DIR, "confusion_matrix.png")):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Malicious"])
    fig, ax = plt.subplots(figsize=(5, 5))
    disp.plot(ax=ax, cmap="Blues", values_format="d")
    ax.set_title("Confusion Matrix -- Tuned Decision Tree")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_roc(model, X_test, y_test, out_path=os.path.join(OUT_DIR, "roc_curve.png")):
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f"Tuned DT (AUC = {auc:.3f})")
    plt.plot([0, 1], [0, 1], "--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve -- Tuned Decision Tree (All Features)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_feature_importance(model, feature_names, out_path=os.path.join(OUT_DIR, "feature_importance.png")):
    importances = model.feature_importances_
    order = np.argsort(importances)[::-1]
    plt.figure(figsize=(9, max(6, 0.28 * len(order))))
    plt.barh([feature_names[i] for i in order][::-1], importances[order][::-1], color="teal")
    plt.xlabel("Gini Importance")
    plt.title(f"Decision Tree Feature Importance -- ALL {len(feature_names)} Features Used")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_tree_structure(model, feature_names, out_path=os.path.join(OUT_DIR, "tree_structure.png"), max_depth_display=3):
    plt.figure(figsize=(20, 10))
    plot_tree(model, feature_names=feature_names, class_names=["Normal", "Malicious"],
              filled=True, max_depth=max_depth_display, fontsize=8)
    plt.title("Tuned Decision Tree -- top levels")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


# ---------------------------------------------------------------------
# 7. MAIN
# ---------------------------------------------------------------------
if __name__ == "__main__":
    # >>>> EDIT THESE PATHS to point at your local UNSW-NB15 files <<<<
    TRAIN_CSV = "UNSW_NB15_training-set.csv"
    TEST_CSV = "UNSW_NB15_testing-set.csv"

    print("Loading data...")
    df = load_unsw_nb15(train_path=TRAIN_CSV, test_path=TEST_CSV)

    print("Preprocessing (no feature selection -- all columns kept)...")
    X, y = preprocess(df)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )

    print("\nHyperparameter tuning (RandomizedSearch -> GridSearch)...")
    t0 = time.time()
    best_model, best_params = tune_decision_tree(X_train, y_train)
    print(f"Tuning finished in {time.time() - t0:.1f}s")

    print("\n5-fold CV evaluation of the tuned model on the FULL dataset...")
    cv_results = evaluate(best_model, X, y, cv_splits=5)
    for k, (m, s) in cv_results.items():
        print(f"{k}: {m:.4f} +/- {s:.4f}")

    print("\nBuilding comparison table against the 3 reference papers...")
    comp_df = build_comparison_table(cv_results)

    print("\nGenerating graphs...")
    best_model.fit(X_train, y_train)  # final fit for the plots below
    plot_comparison_bar(comp_df)
    plot_confusion_matrix(best_model, X_test, y_test)
    plot_roc(best_model, X_test, y_test)
    plot_feature_importance(best_model, X.columns.tolist())
    plot_tree_structure(best_model, X.columns.tolist())

    print(f"\nAll outputs saved to ./{OUT_DIR}/")

# CatBOOST

In [ ]:
"""
CatBoost (Binary Classification) on UNSW-NB15 -- FULL FEATURE SET, GPU-accelerated
========================================================================
Same preprocessing pipeline as xgboost_unsw_nb15_full_features.py
(NO feature selection -- all original columns are kept). Built to be a
directly comparable companion script: same train/test split logic, same
scaler-fit-on-train-only rule, same 5-fold CV protocol -- only the model
and its native hyperparameter space change.

>>> IMPORTANT (Kaggle): this only actually uses a GPU if the notebook's
>>> accelerator is turned on: Settings (right sidebar) -> Accelerator ->
>>> GPU T4 x2 (or P100). CatBoost's GPU check differs from XGBoost's --
>>> it raises a CatBoostError rather than emitting a fallback warning --
>>> so the detection function below is written specifically for that.

Reference results from the 3 papers (these are the numbers to beat --
note they were reported as each paper's XGBoost result, since none of
the three published a CatBoost number for UNSW-NB15; the goal here is
to have CatBoost outperform the best *reported* figure in each paper,
not to match a CatBoost-specific baseline that doesn't exist in them):

  [1] Alkhater, N. (2026). Computers, 15(5), 285.
      -> XGBoost (default config, 5-fold CV, 48 features):
         Acc=0.97+-0.005, Prec=0.96+-0.006, Recall=0.96+-0.005,
         F1=0.96+-0.006, AUC=0.980+-0.003

  [2] Kasongo, S.M. & Sun, Y. (2020). J Big Data, 7:105.
      -> XGBoost in this paper is used ONLY to compute feature-importance
         scores for feature selection -- not one of the 5 final classifiers.
         Excluded from the numeric comparison (marked N/A).

  [3] Mohale, V.Z. & Obagbuwa, I.C. (2025). Front. Comput. Sci., 7:1520741.
      -> XGBoost: Acc=86.87%, Precision=0.85, Recall=0.88, F1=0.86,
         ROC-AUC=0.93, FPR=0.08, FNR=0.11

WHY THIS VERSION IS FASTER (without touching final-model quality):
  1. GPU acceleration (task_type="GPU") instead of CPU.
  2. Hyperparameter SEARCH runs on a stratified ~50k-row subsample of the
     training data -- same rationale as the XGBoost script: rankings
     transfer from a representative sample. The FINAL model is still
     trained on 100% of the data.
  3. Search-level n_jobs=1: a single GPU means parallel sklearn workers
     would just contend for the same device.
  4. A genuine early-stopping pass (Stage 3) using CatBoost's native
     use_best_model=True replaces guessing "iterations" from a coarse
     grid -- this typically *improves* generalization rather than
     trading it away.

HOW TO USE
----------
1. Get UNSW_NB15_training-set.csv / UNSW_NB15_testing-set.csv locally.
2. Edit the paths in the __main__ block if needed.
3. Run: python catboost_unsw_nb15_full_features.py
4. Outputs land in ./outputs/ (kept under *_catboost.* filenames).

Requires: pip install catboost   (use --break-system-packages if needed)
"""

import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
)
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
)

from catboost import CatBoostClassifier, CatBoostError

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

SEARCH_SAMPLE_SIZE = 50_000  # rows used ONLY for hyperparameter search, not final training


# ---------------------------------------------------------------------
# 0. GPU DETECTION (explicit, not silent)
# ---------------------------------------------------------------------
def detect_and_configure_gpu():
    """
    Tries a tiny real fit with task_type='GPU' and catches the CatBoostError
    CatBoost raises when no CUDA-capable device is visible. Prints a clear
    status line either way, mirroring the XGBoost script's behavior, so you
    always know what actually ran instead of silently falling back.
    """
    dummy_X = np.random.rand(200, 5)
    dummy_y = np.random.randint(0, 2, 200)

    try:
        probe = CatBoostClassifier(
            task_type="GPU", devices="0", iterations=5, verbose=False
        )
        probe.fit(dummy_X, dummy_y)
        print("=" * 60)
        print("GPU DETECTED -- running with task_type='GPU'.")
        print("=" * 60)
        return {"task_type": "GPU", "devices": "0"}
    except CatBoostError as e:
        print("=" * 60)
        print("GPU NOT DETECTED -- running on CPU (task_type='CPU').")
        print(f"(CatBoost reported: {str(e)[:120]})")
        print("If you're on Kaggle: Settings -> Accelerator -> GPU T4 x2")
        print("=" * 60)
        return {"task_type": "CPU"}


# ---------------------------------------------------------------------
# 1. LOAD DATA  (identical to the XGBoost / Decision Tree scripts)
# ---------------------------------------------------------------------
def load_unsw_nb15(train_path=None, test_path=None):
    df_train = pd.read_csv(
        "/kaggle/input/datasets/ajeetkumar20/unsw-nb15-v1-1/UNSW_NB15_training-set.csv"
    )
    df_test = pd.read_csv(
        "/kaggle/input/datasets/ajeetkumar20/unsw-nb15-v1-1/UNSW_NB15_testing-set.csv"
    )
    return df_train, df_test


# ---------------------------------------------------------------------
# 2. PREPROCESSING  (NO FEATURE SELECTION -- byte-for-byte identical
#    logic to the XGBoost script, so results are directly comparable:
#    encoders/scaler fit ONLY on train, applied to test)
# ---------------------------------------------------------------------
def preprocess_train_test(df_train, df_test):
    """
    Preprocess train and test separately.
    Encoders and scaler are FIT ONLY on training data.
    """
    train = df_train.copy()
    test = df_test.copy()

    # --------------------------------------------------
    # Remove ID columns
    # --------------------------------------------------
    for id_col in ["id", "ID", "Id"]:
        if id_col in train.columns:
            train.drop(columns=[id_col], inplace=True)
        if id_col in test.columns:
            test.drop(columns=[id_col], inplace=True)

    # --------------------------------------------------
    # Separate labels
    # --------------------------------------------------
    y_train = train["label"].astype(int)
    y_test = test["label"].astype(int)
    train.drop(columns=["label"], inplace=True)
    test.drop(columns=["label"], inplace=True)

    # --------------------------------------------------
    # Remove attack_cat (label leakage)
    # --------------------------------------------------
    if "attack_cat" in train.columns:
        train.drop(columns=["attack_cat"], inplace=True)
    if "attack_cat" in test.columns:
        test.drop(columns=["attack_cat"], inplace=True)

    # --------------------------------------------------
    # Label Encoding
    # (Note: CatBoost can natively handle raw categorical columns via
    # cat_features=[...] without this step, which is normally one of its
    # selling points. We deliberately keep the identical LabelEncoder +
    # MinMaxScaler pipeline here instead, so the comparison against the
    # XGBoost run -- and against the papers' reported numbers -- is on
    # exactly the same inputs, not on a pipeline CatBoost is naturally
    # better suited to.)
    # --------------------------------------------------
    encoders = {}
    categorical_cols = train.select_dtypes(include=["object"]).columns
    for col in categorical_cols:
        le = LabelEncoder()
        combined = pd.concat([
            train[col].astype(str),
            test[col].astype(str)
        ])
        le.fit(combined)
        train[col] = le.transform(train[col].astype(str))
        test[col] = le.transform(test[col].astype(str))
        encoders[col] = le

    # --------------------------------------------------
    # Scale ONLY using training statistics
    # --------------------------------------------------
    scaler = MinMaxScaler()
    X_train = pd.DataFrame(
        scaler.fit_transform(train),
        columns=train.columns
    )
    X_test = pd.DataFrame(
        scaler.transform(test),
        columns=test.columns
    )

    print(f"Training features : {X_train.shape}")
    print(f"Testing features  : {X_test.shape}")

    return X_train, X_test, y_train, y_test, scaler, encoders


# ---------------------------------------------------------------------
# 3. HYPERPARAMETER TUNING (3-stage: RandomizedSearch -> GridSearch ->
#    early-stopping "iterations" refinement)
# ---------------------------------------------------------------------
def make_search_subsample(X_train, y_train, sample_size=SEARCH_SAMPLE_SIZE, random_state=RANDOM_STATE):
    """Stratified subsample used ONLY for the hyperparameter search phase.
    The final model is always trained on the full X_train, y_train."""
    if len(X_train) <= sample_size:
        return X_train, y_train
    X_search, _, y_search, _ = train_test_split(
        X_train, y_train, train_size=sample_size, stratify=y_train, random_state=random_state
    )
    print(f"Hyperparameter search will use a stratified subsample: "
          f"{len(X_search)} / {len(X_train)} rows ({100 * len(X_search) / len(X_train):.1f}%)")
    return X_search, y_search


def tune_catboost(X_train, y_train, gpu_params):
    X_search, y_search = make_search_subsample(X_train, y_train)

    base_cb = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="Logloss",
        random_seed=RANDOM_STATE,
        verbose=False,
        **gpu_params,
    )

    # Fewer folds for the SEARCH phase only -- 3-fold is plenty to rank
    # hyperparameters on a subsample; the final reported metrics later
    # still use full 5-fold CV on the complete dataset.
    search_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    # ---- Stage 1: broad RandomizedSearchCV ----
    param_dist = {
        # iterations just needs to be "big enough" here -- Stage 3 below
        # finds the real, precisely-tuned value via early stopping.
        "iterations": [200, 300, 500, 800],
        "depth": [4, 5, 6, 7, 8, 9, 10],
        "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08, 0.1, 0.15, 0.2],
        "l2_leaf_reg": [1, 3, 5, 7, 9, 12, 15],
        "border_count": [32, 64, 128, 254],
        "bagging_temperature": [0, 0.2, 0.5, 1.0, 2.0],
        "random_strength": [0, 0.5, 1, 2, 5],
    }

    t0 = time.time()
    rand_search = RandomizedSearchCV(
        base_cb, param_distributions=param_dist, n_iter=40,
        scoring="f1", cv=search_cv,
        n_jobs=1,  # single GPU -- parallel workers would just contend for it
        random_state=RANDOM_STATE, verbose=1
    )
    rand_search.fit(X_search, y_search)
    best_rand = rand_search.best_params_
    print(f"Stage-1 (Randomized) best params: {best_rand}")
    print(f"Stage-1 took {time.time() - t0:.1f}s")

    # ---- Stage 2: narrow GridSearchCV around the best region ----
    def widen(v, lo_bound, step, as_int=True):
        vals = {v - step, v, v + step}
        vals = {max(lo_bound, x) for x in vals}
        return sorted({int(x) if as_int else round(x, 4) for x in vals})

    grid = {
        "iterations": widen(best_rand["iterations"], 100, 150),
        "depth": widen(best_rand["depth"], 2, 1),
        "learning_rate": widen(best_rand["learning_rate"], 0.005, 0.02, as_int=False),
        "l2_leaf_reg": widen(best_rand["l2_leaf_reg"], 1, 2),
        "border_count": [best_rand["border_count"]],
        "bagging_temperature": [best_rand["bagging_temperature"]],
        "random_strength": [best_rand["random_strength"]],
    }

    t0 = time.time()
    grid_search = GridSearchCV(
        base_cb, param_grid=grid, scoring="f1", cv=search_cv, n_jobs=1, verbose=1
    )
    grid_search.fit(X_search, y_search)
    best_params = grid_search.best_params_
    print(f"Stage-2 (Grid) best params: {best_params}")
    print(f"Stage-2 best CV F1 (on subsample): {grid_search.best_score_:.4f}")
    print(f"Stage-2 took {time.time() - t0:.1f}s")

    # ---- Stage 3: early-stopping refinement of "iterations", on the
    #      FULL training set (not the subsample) so the final boosting-
    #      round count is calibrated to the real data scale. ----
    t0 = time.time()
    X_fit, X_es_val, y_fit, y_es_val = train_test_split(
        X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE
    )

    es_params = {k: v for k, v in best_params.items() if k != "iterations"}
    es_model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="Logloss",
        random_seed=RANDOM_STATE,
        iterations=3000,            # generous ceiling -- early stopping decides the real count
        early_stopping_rounds=50,
        use_best_model=True,
        verbose=False,
        **es_params,
        **gpu_params,
    )
    es_model.fit(X_fit, y_fit, eval_set=(X_es_val, y_es_val))
    final_iterations = es_model.get_best_iteration() + 1
    print(f"Early-stopping found optimal iterations = {final_iterations} "
          f"(ceiling was 3000, patience 50 rounds)")
    print(f"Stage-3 took {time.time() - t0:.1f}s")

    # ---- Build the final, fully-specified model ----
    final_params = dict(best_params)
    final_params["iterations"] = final_iterations
    final_model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="Logloss",
        random_seed=RANDOM_STATE,
        verbose=False,
        **final_params,
        **gpu_params,
    )

    print(f"\nFinal tuned hyperparameters: {final_params}")
    return final_model, final_params


# ---------------------------------------------------------------------
# 4. EVALUATION (5-fold stratified CV -- same protocol as the XGBoost
#    script and as reference paper [1], so results are directly comparable)
# ---------------------------------------------------------------------
def evaluate(model, X, y, cv_splits=5):
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
    accs, precs, recs, f1s, aucs = [], [], [], [], []
    t0 = time.time()
    for fold_i, (train_idx, test_idx) in enumerate(cv.split(X, y), 1):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        y_proba = model.predict_proba(X_te)[:, 1]

        accs.append(accuracy_score(y_te, y_pred))
        precs.append(precision_score(y_te, y_pred))
        recs.append(recall_score(y_te, y_pred))
        f1s.append(f1_score(y_te, y_pred))
        aucs.append(roc_auc_score(y_te, y_proba))
        print(f"  Fold {fold_i}/{cv_splits} done ({time.time() - t0:.1f}s elapsed)")

    return {
        "Accuracy": (np.mean(accs), np.std(accs)),
        "Precision": (np.mean(precs), np.std(precs)),
        "Recall": (np.mean(recs), np.std(recs)),
        "F1-Score": (np.mean(f1s), np.std(f1s)),
        "AUC": (np.mean(aucs), np.std(aucs)),
    }


# ---------------------------------------------------------------------
# 5. COMPARISON TABLE (against the 3 reference papers -- same numbers
#    as the XGBoost script, since these are the figures to beat)
# ---------------------------------------------------------------------
REFERENCE_RESULTS = {
    "Alkhater (2026) - reported XGBoost": {
        "Accuracy": 0.970, "Precision": 0.960, "Recall": 0.960,
        "F1-Score": 0.960, "AUC": 0.980,
    },
    "Kasongo & Sun (2020) - XGBoost (feature-selection only)": {
        "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan,
        "F1-Score": np.nan, "AUC": np.nan,
    },
    "Mohale & Obagbuwa (2025) - reported XGBoost": {
        "Accuracy": 0.8687, "Precision": 0.850, "Recall": 0.880,
        "F1-Score": 0.860, "AUC": 0.930,
    },
}


def build_comparison_table(my_results, out_csv=os.path.join(OUT_DIR, "comparison_table_catboost.csv")):
    rows = []
    for name, metrics in REFERENCE_RESULTS.items():
        row = {"Model": name}
        row.update(metrics)
        rows.append(row)

    my_row = {"Model": "Proposed Tuned CatBoost (ALL features, no selection)"}
    for k, (mean, _std) in my_results.items():
        my_row[k] = round(mean, 4)
    rows.append(my_row)

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(df.to_string(index=False))
    return df


# ---------------------------------------------------------------------
# 6. GRAPHS
# ---------------------------------------------------------------------
def plot_comparison_bar(df, out_path=os.path.join(OUT_DIR, "comparison_bar_catboost.png")):
    metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]
    x = np.arange(len(df))
    width = 0.2

    fig, ax = plt.subplots(figsize=(11, 6))
    for i, m in enumerate(metrics):
        ax.bar(x + i * width, df[m], width, label=m)
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(df["Model"], rotation=20, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title("CatBoost -- Proposed Model vs. Reference Papers")
    ax.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_confusion_matrix(model, X_test, y_test, out_path=os.path.join(OUT_DIR, "confusion_matrix_catboost.png")):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Malicious"])
    fig, ax = plt.subplots(figsize=(5, 5))
    disp.plot(ax=ax, cmap="Purples", values_format="d")
    ax.set_title("Confusion Matrix -- Tuned CatBoost")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_roc(model, X_test, y_test, out_path=os.path.join(OUT_DIR, "roc_curve_catboost.png")):
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)

    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, color="purple", label=f"Tuned CatBoost (AUC = {auc:.3f})")
    plt.plot([0, 1], [0, 1], "--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve -- Tuned CatBoost (All Features)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_feature_importance(model, feature_names, out_path=os.path.join(OUT_DIR, "feature_importance_catboost.png")):
    importances = model.get_feature_importance()
    order = np.argsort(importances)[::-1]

    plt.figure(figsize=(9, max(6, 0.28 * len(order))))
    plt.barh([feature_names[i] for i in order][::-1], importances[order][::-1], color="darkviolet")
    plt.xlabel("PredictionValuesChange Importance")
    plt.title(f"CatBoost Feature Importance -- ALL {len(feature_names)} Features Used")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


# ---------------------------------------------------------------------
# 7. MAIN
# ---------------------------------------------------------------------
if __name__ == "__main__":
    TRAIN_CSV = "UNSW_NB15_training-set.csv"
    TEST_CSV = "UNSW_NB15_testing-set.csv"

    pipeline_start = time.time()

    gpu_params = detect_and_configure_gpu()

    print("\nLoading data...")
    df_train, df_test = load_unsw_nb15(train_path=TRAIN_CSV, test_path=TEST_CSV)

    print("Preprocessing (no feature selection -- all columns kept, "
          "encoders/scaler fit on train only)...")
    X_train, X_test, y_train, y_test, scaler, encoders = preprocess_train_test(df_train, df_test)

    # Full dataset (train+test recombined) used for the final reported
    # 5-fold CV, exactly as in the XGBoost script.
    X_full = pd.concat([X_train, X_test], axis=0).reset_index(drop=True)
    y_full = pd.concat([y_train, y_test], axis=0).reset_index(drop=True)

    print("\nHyperparameter tuning (Random -> Grid -> early-stopping refinement)...")
    t0 = time.time()
    best_model, best_params = tune_catboost(X_train, y_train, gpu_params)
    print(f"\nTotal tuning time: {time.time() - t0:.1f}s")

    print("\n5-fold CV evaluation of the tuned model on the FULL dataset...")
    t0 = time.time()
    cv_results = evaluate(best_model, X_full, y_full, cv_splits=5)
    print(f"5-fold evaluation took {time.time() - t0:.1f}s")
    for k, (m, s) in cv_results.items():
        print(f"{k}: {m:.4f} +/- {s:.4f}")

    print("\nBuilding comparison table against the 3 reference papers...")
    comp_df = build_comparison_table(cv_results)

    print("\nGenerating graphs...")
    best_model.fit(X_train, y_train)  # final fit for the plots below
    plot_comparison_bar(comp_df)
    plot_confusion_matrix(best_model, X_test, y_test)
    plot_roc(best_model, X_test, y_test)
    plot_feature_importance(best_model, X_full.columns.tolist())

    print(f"\nAll outputs saved to ./{OUT_DIR}/ (with *_catboost suffix)")
    print(f"\nTOTAL PIPELINE TIME: {time.time() - pipeline_start:.1f}s")

# XGBoost

In [ ]:
"""
XGBoost (Binary Classification) on UNSW-NB15 -- FULL FEATURE SET, GPU-accelerated
========================================================================
Same preprocessing pipeline as decision_tree_unsw_nb15_full_features.py
(NO feature selection -- all original columns are kept).

>>> IMPORTANT (Kaggle): this only actually uses a GPU if the notebook's
>>> accelerator is turned on: Settings (right sidebar) -> Accelerator ->
>>> GPU T4 x2 (or P100). The code below auto-detects this at runtime and
>>> prints clearly whether it's really running on GPU or falling back to
>>> CPU -- it will NOT silently pretend to be fast if the accelerator
>>> isn't enabled.

Reference XGBoost numbers from the 3 papers:

  [1] Alkhater, N. (2026). Computers, 15(5), 285.
      -> XGBoost (default config, 5-fold CV, 48 features):
         Acc=0.97+-0.005, Prec=0.96+-0.006, Recall=0.96+-0.005,
         F1=0.96+-0.006, AUC=0.980+-0.003

  [2] Kasongo, S.M. & Sun, Y. (2020). J Big Data, 7:105.
      -> XGBoost in this paper is used ONLY to compute feature-importance
         scores for feature selection -- not one of the 5 final classifiers.
         Excluded from the numeric comparison (marked N/A).

  [3] Mohale, V.Z. & Obagbuwa, I.C. (2025). Front. Comput. Sci., 7:1520741.
      -> XGBoost: Acc=86.87%, Precision=0.85, Recall=0.88, F1=0.86,
         ROC-AUC=0.93, FPR=0.08, FNR=0.11

WHY THIS VERSION IS FASTER (without touching final-model quality):
  1. GPU acceleration (device="cuda") instead of CPU-only "hist".
  2. Hyperparameter SEARCH runs on a stratified ~50k-row subsample of the
     training data -- hyperparameter rankings transfer from a
     representative sample, so this doesn't need the full ~200k rows to
     find the right region. The FINAL model is still trained on 100% of
     the data; only the expensive iterative search phase is subsampled.
  3. Search-level n_jobs=1: with a single GPU, parallel sklearn workers
     would just contend for the same device instead of speeding anything
     up.
  4. A genuine early-stopping pass (Stage 3) replaces guessing
     n_estimators from a coarse grid -- this typically *improves*
     generalization (stops exactly at the least-overfit point) rather
     than trading it away.

HOW TO USE
----------
1. Get UNSW_NB15_training-set.csv / UNSW_NB15_testing-set.csv locally.
2. Edit the paths in the __main__ block if needed.
3. Run: python xgboost_unsw_nb15_full_features.py
4. Outputs land in ./outputs/ (kept under *_xgboost.* filenames).

Requires: pip install xgboost   (use --break-system-packages if needed)
"""

import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
)
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

SEARCH_SAMPLE_SIZE = 50_000  # rows used ONLY for hyperparameter search, not final training


# ---------------------------------------------------------------------
# 0. GPU DETECTION (explicit, not silent)
# ---------------------------------------------------------------------
def detect_and_configure_gpu():
    """
    Tries a tiny real fit with device='cuda' and inspects whether XGBoost
    had to fall back to CPU. Returns a dict of params to merge into every
    XGBClassifier() constructor call. Prints a clear status line either way
    so you always know what actually ran, instead of a buried warning.
    """
    dummy_X = np.random.rand(200, 5)
    dummy_y = np.random.randint(0, 2, 200)

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        probe = XGBClassifier(
            tree_method="hist", device="cuda", n_estimators=5, verbosity=0
        )
        probe.fit(dummy_X, dummy_y)

    fell_back_to_cpu = any("couldn't find any available GPU" in str(w.message)
                            or "No visible GPU" in str(w.message) for w in caught)

    if fell_back_to_cpu:
        print("=" * 60)
        print("GPU NOT DETECTED -- running on CPU (tree_method='hist').")
        print("If you're on Kaggle: Settings -> Accelerator -> GPU T4 x2")
        print("=" * 60)
        return {"tree_method": "hist", "device": "cpu"}
    else:
        print("=" * 60)
        print("GPU DETECTED -- running with device='cuda'.")
        print("=" * 60)
        return {"tree_method": "hist", "device": "cuda"}


# ---------------------------------------------------------------------
# 1. LOAD DATA  (identical to the Decision Tree script)
# ---------------------------------------------------------------------
def load_unsw_nb15(train_path=None, test_path=None):

    df_train = pd.read_csv(
        "/kaggle/input/datasets/ajeetkumar20/unsw-nb15-v1-1/UNSW_NB15_training-set.csv"
    )

    df_test = pd.read_csv(
        "/kaggle/input/datasets/ajeetkumar20/unsw-nb15-v1-1/UNSW_NB15_testing-set.csv"
    )

    return df_train, df_test


# ---------------------------------------------------------------------
# 2. PREPROCESSING  (NO FEATURE SELECTION -- identical to DT script,
#    so both models are compared on exactly the same inputs)
# ---------------------------------------------------------------------
# def preprocess(df):
#     df = df.copy()

#     for id_col in ["id", "ID", "Id"]:
#         if id_col in df.columns:
#             df = df.drop(columns=[id_col])

#     if "label" in df.columns:
#         y = df["label"].astype(int)
#         df = df.drop(columns=["label"])
#     elif "Label" in df.columns:
#         y = df["Label"].astype(int)
#         df = df.drop(columns=["Label"])
#     else:
#         raise ValueError("Could not find a binary 'label' column in the data.")

#     if "attack_cat" in df.columns:
#         df = df.drop(columns=["attack_cat"])  # label-leak column, not a feature drop

#     cat_cols = [c for c in ["proto", "service", "state"] if c in df.columns]
#     for c in cat_cols:
#         df[c] = LabelEncoder().fit_transform(df[c].astype(str))

#     for c in df.columns:
#         if df[c].dtype == object:
#             df[c] = LabelEncoder().fit_transform(df[c].astype(str))

#     scaler = MinMaxScaler()
#     X = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

#     print(f"Final feature matrix: {X.shape[1]} features "
#           f"(ALL features retained -- no selection/reduction applied)")
#     return X, y

def preprocess_train_test(df_train, df_test):
    """
    Preprocess train and test separately.
    Encoders and scaler are FIT ONLY on training data.
    """

    train = df_train.copy()
    test = df_test.copy()

    # --------------------------------------------------
    # Remove ID columns
    # --------------------------------------------------
    for id_col in ["id", "ID", "Id"]:
        if id_col in train.columns:
            train.drop(columns=[id_col], inplace=True)
        if id_col in test.columns:
            test.drop(columns=[id_col], inplace=True)

    # --------------------------------------------------
    # Separate labels
    # --------------------------------------------------
    y_train = train["label"].astype(int)
    y_test = test["label"].astype(int)

    train.drop(columns=["label"], inplace=True)
    test.drop(columns=["label"], inplace=True)

    # --------------------------------------------------
    # Remove attack_cat (label leakage)
    # --------------------------------------------------
    if "attack_cat" in train.columns:
        train.drop(columns=["attack_cat"], inplace=True)

    if "attack_cat" in test.columns:
        test.drop(columns=["attack_cat"], inplace=True)

    # --------------------------------------------------
    # Label Encoding
    # --------------------------------------------------
    encoders = {}

    categorical_cols = train.select_dtypes(include=["object"]).columns

    for col in categorical_cols:

        le = LabelEncoder()

        combined = pd.concat([
            train[col].astype(str),
            test[col].astype(str)
        ])

        le.fit(combined)

        train[col] = le.transform(train[col].astype(str))
        test[col] = le.transform(test[col].astype(str))

        encoders[col] = le

    # --------------------------------------------------
    # Scale ONLY using training statistics
    # --------------------------------------------------
    scaler = MinMaxScaler()

    X_train = pd.DataFrame(
        scaler.fit_transform(train),
        columns=train.columns
    )

    X_test = pd.DataFrame(
        scaler.transform(test),
        columns=test.columns
    )

    print(f"Training features : {X_train.shape}")
    print(f"Testing features  : {X_test.shape}")

    return X_train, X_test, y_train, y_test, scaler, encoders


# ---------------------------------------------------------------------
# 3. HYPERPARAMETER TUNING (3-stage: RandomizedSearch -> GridSearch ->
#    early-stopping n_estimators refinement)
# ---------------------------------------------------------------------
def make_search_subsample(X_train, y_train, sample_size=SEARCH_SAMPLE_SIZE, random_state=RANDOM_STATE):
    """Stratified subsample used ONLY for the hyperparameter search phase.
    The final model is always trained on the full X_train, y_train."""
    if len(X_train) <= sample_size:
        return X_train, y_train
    X_search, _, y_search, _ = train_test_split(
        X_train, y_train, train_size=sample_size, stratify=y_train, random_state=random_state
    )
    print(f"Hyperparameter search will use a stratified subsample: "
          f"{len(X_search)} / {len(X_train)} rows ({100 * len(X_search) / len(X_train):.1f}%)")
    return X_search, y_search


def tune_xgboost(X_train, y_train, gpu_params):
    X_search, y_search = make_search_subsample(X_train, y_train)

    base_xgb = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        **gpu_params,
    )
    # Fewer folds for the SEARCH phase only -- 3-fold is plenty to rank
    # hyperparameters on a subsample; the final reported metrics later
    # still use full 5-fold CV on the complete dataset.
    search_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    # ---- Stage 1: broad RandomizedSearchCV ----
    param_dist = {
        # n_estimators just needs to be "big enough" here -- Stage 3 below
        # finds the real, precisely-tuned value via early stopping.
        "n_estimators": [100, 150, 200, 300, 400],
        "max_depth": [3, 4, 5, 6, 7, 8, 9, 10],
        "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08, 0.1, 0.15, 0.2],
        "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
        "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
        "gamma": [0, 0.1, 0.2, 0.3, 0.5, 1.0],
        "min_child_weight": [1, 2, 3, 5, 7, 10],
        "reg_alpha": [0, 0.001, 0.01, 0.1, 1],
        "reg_lambda": [0.5, 1, 1.5, 2, 3, 5],
    }

    t0 = time.time()
    rand_search = RandomizedSearchCV(
        base_xgb, param_distributions=param_dist, n_iter=40,
        scoring="f1", cv=search_cv,
        n_jobs=1,  # single GPU -- parallel workers would just contend for it
        random_state=RANDOM_STATE, verbose=1
    )
    rand_search.fit(X_search, y_search)
    best_rand = rand_search.best_params_
    print(f"Stage-1 (Randomized) best params: {best_rand}")
    print(f"Stage-1 took {time.time() - t0:.1f}s")

    # ---- Stage 2: narrow GridSearchCV around the best region ----
    def widen(v, lo_bound, step, as_int=True):
        vals = {v - step, v, v + step}
        vals = {max(lo_bound, x) for x in vals}
        return sorted({int(x) if as_int else round(x, 4) for x in vals})

    grid = {
        "n_estimators": widen(best_rand["n_estimators"], 50, 100),
        "max_depth": widen(best_rand["max_depth"], 2, 1),
        "learning_rate": widen(best_rand["learning_rate"], 0.005, 0.02, as_int=False),
        "subsample": [best_rand["subsample"]],
        "colsample_bytree": [best_rand["colsample_bytree"]],
        "gamma": [best_rand["gamma"]],
        "min_child_weight": widen(best_rand["min_child_weight"], 1, 1),
        "reg_alpha": [best_rand["reg_alpha"]],
        "reg_lambda": [best_rand["reg_lambda"]],
    }

    t0 = time.time()
    grid_search = GridSearchCV(
        base_xgb, param_grid=grid, scoring="f1", cv=search_cv, n_jobs=1, verbose=1
    )
    grid_search.fit(X_search, y_search)
    best_params = grid_search.best_params_
    print(f"Stage-2 (Grid) best params: {best_params}")
    print(f"Stage-2 best CV F1 (on subsample): {grid_search.best_score_:.4f}")
    print(f"Stage-2 took {time.time() - t0:.1f}s")

    # ---- Stage 3: early-stopping refinement of n_estimators, on the
    #      FULL training set (not the subsample) so the final boosting-
    #      round count is calibrated to the real data scale. ----
    t0 = time.time()
    X_fit, X_es_val, y_fit, y_es_val = train_test_split(
        X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE
    )

    es_params = {k: v for k, v in best_params.items() if k != "n_estimators"}
    es_model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_estimators=2000,          # generous ceiling -- early stopping decides the real count
        early_stopping_rounds=40,
        **es_params,
        **gpu_params,
    )
    es_model.fit(X_fit, y_fit, eval_set=[(X_es_val, y_es_val)], verbose=False)
    final_n_estimators = es_model.best_iteration + 1
    print(f"Early-stopping found optimal n_estimators = {final_n_estimators} "
          f"(ceiling was 2000, patience 40 rounds)")
    print(f"Stage-3 took {time.time() - t0:.1f}s")

    # ---- Build the final, fully-specified model ----
    final_params = dict(best_params)
    final_params["n_estimators"] = final_n_estimators
    final_model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        **final_params,
        **gpu_params,
    )

    print(f"\nFinal tuned hyperparameters: {final_params}")
    return final_model, final_params


# ---------------------------------------------------------------------
# 4. EVALUATION (5-fold stratified CV -- same protocol as the DT script
#    and as reference paper [1], so results are directly comparable)
# ---------------------------------------------------------------------
def evaluate(model, X, y, cv_splits=5):
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
    accs, precs, recs, f1s, aucs = [], [], [], [], []

    t0 = time.time()
    for fold_i, (train_idx, test_idx) in enumerate(cv.split(X, y), 1):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        y_proba = model.predict_proba(X_te)[:, 1]

        accs.append(accuracy_score(y_te, y_pred))
        precs.append(precision_score(y_te, y_pred))
        recs.append(recall_score(y_te, y_pred))
        f1s.append(f1_score(y_te, y_pred))
        aucs.append(roc_auc_score(y_te, y_proba))
        print(f"  Fold {fold_i}/{cv_splits} done ({time.time() - t0:.1f}s elapsed)")

    return {
        "Accuracy": (np.mean(accs), np.std(accs)),
        "Precision": (np.mean(precs), np.std(precs)),
        "Recall": (np.mean(recs), np.std(recs)),
        "F1-Score": (np.mean(f1s), np.std(f1s)),
        "AUC": (np.mean(aucs), np.std(aucs)),
    }


# ---------------------------------------------------------------------
# 5. COMPARISON TABLE (against the reported XGBoost numbers)
# ---------------------------------------------------------------------
XGBOOST_REFERENCE_RESULTS = {
    "Alkhater (2026) - XGBoost": {
        "Accuracy": 0.970, "Precision": 0.960, "Recall": 0.960,
        "F1-Score": 0.960, "AUC": 0.980,
    },
    "Kasongo & Sun (2020) - XGBoost": {
        "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan,
        "F1-Score": np.nan, "AUC": np.nan,
    },
    "Mohale & Obagbuwa (2025) - XGBoost": {
        "Accuracy": 0.8687, "Precision": 0.850, "Recall": 0.880,
        "F1-Score": 0.860, "AUC": 0.930,
    },
}


def build_comparison_table(my_results, out_csv=os.path.join(OUT_DIR, "comparison_table_xgboost.csv")):
    rows = []
    for name, metrics in XGBOOST_REFERENCE_RESULTS.items():
        row = {"Model": name}
        row.update(metrics)
        rows.append(row)

    my_row = {"Model": "Proposed Tuned XGBoost (ALL features, no selection)"}
    for k, (mean, _std) in my_results.items():
        my_row[k] = round(mean, 4)
    rows.append(my_row)

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(df.to_string(index=False))
    return df


# ---------------------------------------------------------------------
# 6. GRAPHS
# ---------------------------------------------------------------------
def plot_comparison_bar(df, out_path=os.path.join(OUT_DIR, "comparison_bar_xgboost.png")):
    metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]
    x = np.arange(len(df))
    width = 0.2
    fig, ax = plt.subplots(figsize=(11, 6))
    for i, m in enumerate(metrics):
        ax.bar(x + i * width, df[m], width, label=m)
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(df["Model"], rotation=20, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title("XGBoost -- Proposed Model vs. Reference Papers")
    ax.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_confusion_matrix(model, X_test, y_test, out_path=os.path.join(OUT_DIR, "confusion_matrix_xgboost.png")):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Malicious"])
    fig, ax = plt.subplots(figsize=(5, 5))
    disp.plot(ax=ax, cmap="Greens", values_format="d")
    ax.set_title("Confusion Matrix -- Tuned XGBoost")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_roc(model, X_test, y_test, out_path=os.path.join(OUT_DIR, "roc_curve_xgboost.png")):
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, color="green", label=f"Tuned XGBoost (AUC = {auc:.3f})")
    plt.plot([0, 1], [0, 1], "--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve -- Tuned XGBoost (All Features)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_feature_importance(model, feature_names, out_path=os.path.join(OUT_DIR, "feature_importance_xgboost.png")):
    importances = model.feature_importances_
    order = np.argsort(importances)[::-1]
    plt.figure(figsize=(9, max(6, 0.28 * len(order))))
    plt.barh([feature_names[i] for i in order][::-1], importances[order][::-1], color="darkgreen")
    plt.xlabel("Gain-based Importance")
    plt.title(f"XGBoost Feature Importance -- ALL {len(feature_names)} Features Used")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


# ---------------------------------------------------------------------
# 7. MAIN
# ---------------------------------------------------------------------
if __name__ == "__main__":
    TRAIN_CSV = "UNSW_NB15_training-set.csv"
    TEST_CSV = "UNSW_NB15_testing-set.csv"

    pipeline_start = time.time()

    gpu_params = detect_and_configure_gpu()

    print("\nLoading data...")
    df = load_unsw_nb15(train_path=TRAIN_CSV, test_path=TEST_CSV)

    print("Preprocessing (no feature selection -- all columns kept)...")
    X, y = preprocess(df)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )

    print("\nHyperparameter tuning (Random -> Grid -> early-stopping refinement)...")
    t0 = time.time()
    best_model, best_params = tune_xgboost(X_train, y_train, gpu_params)
    print(f"\nTotal tuning time: {time.time() - t0:.1f}s")

    print("\n5-fold CV evaluation of the tuned model on the FULL dataset...")
    t0 = time.time()
    cv_results = evaluate(best_model, X, y, cv_splits=5)
    print(f"5-fold evaluation took {time.time() - t0:.1f}s")
    for k, (m, s) in cv_results.items():
        print(f"{k}: {m:.4f} +/- {s:.4f}")

    print("\nBuilding comparison table against the 3 reference papers...")
    comp_df = build_comparison_table(cv_results)

    print("\nGenerating graphs...")
    best_model.fit(X_train, y_train)  # final fit for the plots below
    plot_comparison_bar(comp_df)
    plot_confusion_matrix(best_model, X_test, y_test)
    plot_roc(best_model, X_test, y_test)
    plot_feature_importance(best_model, X.columns.tolist())

    print(f"\nAll outputs saved to ./{OUT_DIR}/ (with *_xgboost suffix)")
    print(f"\nTOTAL PIPELINE TIME: {time.time() - pipeline_start:.1f}s")

# MLP

In [ ]:
"""
MLP (Binary Classification) on UNSW-NB15 -- FULL FEATURE SET -- GPU VERSION
============================================================================
Same task/preprocessing/benchmarks as mlp_unsw_nb15_full_features.py, but
built on TensorFlow/Keras so training actually uses your GPU. The plain
sklearn MLPClassifier version is CPU-only regardless of hardware -- that
is why it was slow.

Benchmarks (UNSW-NB15 binary classification):
  [1] Li (2026), Table 4          -> MLP: Acc=0.8578, Prec=0.8432, Recall=0.8896, F1=0.8498
  [2] Mohale & Obagbuwa (2025), Table 4 -> MLP: Acc=0.8598, Prec=0.84, Recall=0.87, F1=0.85, AUC=0.91

TARGET: Accuracy > 0.8598 (the higher of the two baselines).

REQUIREMENTS
------------
pip install tensorflow scikit-learn pandas matplotlib --break-system-packages
(use tensorflow[and-cuda] on Linux with an NVIDIA GPU if TF doesn't pick up
 your CUDA install automatically: pip install "tensorflow[and-cuda]")

HOW TO USE
----------
1. Download UNSW-NB15 (UNSW_NB15_training-set.csv / UNSW_NB15_testing-set.csv).
2. Edit TRAIN_CSV / TEST_CSV in the __main__ block.
3. Run:  python mlp_unsw_nb15_full_features_gpu.py
4. Check ./outputs/ for the comparison table + plots.
"""

import os
import time
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")  # quieter TF logs

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)


# ---------------------------------------------------------------------
# 0. GPU CHECK
# ---------------------------------------------------------------------
def configure_gpu():
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        for gpu in gpus:
            try:
                tf.config.experimental.set_memory_growth(gpu, True)
            except RuntimeError:
                pass  # must be set before GPUs are initialized; safe to ignore if already set
        print(f"GPU(s) detected: {[g.name for g in gpus]} -- training will run on GPU.")
    else:
        print("No GPU detected by TensorFlow. Training will run on CPU.\n"
              "If you have an NVIDIA GPU, check your CUDA/cuDNN install, or try:\n"
              '  pip install "tensorflow[and-cuda]" --break-system-packages')
    return len(gpus) > 0


# ---------------------------------------------------------------------
# 1. LOAD DATA (identical to the reference / sklearn scripts)
# ---------------------------------------------------------------------
def load_unsw_nb15(train_path=None, test_path=None, single_path=None):
    if train_path and test_path:
        df_train = pd.read_csv("/kaggle/input/datasets/ajeetkumar20/unsw-nb15-v1-1/UNSW_NB15_training-set.csv")
        df_test = pd.read_csv("/kaggle/input/datasets/ajeetkumar20/unsw-nb15-v1-1/UNSW_NB15_testing-set.csv")
        df = pd.concat([df_train, df_test], axis=0, ignore_index=True)
    elif single_path:
        df = pd.read_csv(single_path)
    else:
        raise ValueError("Provide either (train_path & test_path) or single_path")
    return df


# ---------------------------------------------------------------------
# 2. PREPROCESSING (NO FEATURE SELECTION -- every column kept)
# ---------------------------------------------------------------------
def preprocess(df):
    df = df.copy()

    for id_col in ["id", "ID", "Id"]:
        if id_col in df.columns:
            df = df.drop(columns=[id_col])

    if "label" in df.columns:
        y = df["label"].astype(int)
        df = df.drop(columns=["label"])
    elif "Label" in df.columns:
        y = df["Label"].astype(int)
        df = df.drop(columns=["Label"])
    else:
        raise ValueError("Could not find a binary 'label' column in the data.")

    if "attack_cat" in df.columns:
        df = df.drop(columns=["attack_cat"])  # label leak, not feature selection

    cat_cols = [c for c in ["proto", "service", "state"] if c in df.columns]
    for c in cat_cols:
        df[c] = LabelEncoder().fit_transform(df[c].astype(str))

    for c in df.columns:
        if df[c].dtype == object:
            df[c] = LabelEncoder().fit_transform(df[c].astype(str))

    scaler = MinMaxScaler()
    X = pd.DataFrame(scaler.fit_transform(df), columns=df.columns).astype("float32")

    print(f"Final feature matrix: {X.shape[1]} features "
          f"(ALL features retained -- no selection/reduction applied)")
    return X, y.astype("float32")


# ---------------------------------------------------------------------
# 3. MODEL BUILDER
# ---------------------------------------------------------------------
def build_model(input_dim, hidden_layers, dropout, l2, learning_rate, activation):
    model = keras.Sequential(name="mlp_ids")
    model.add(keras.Input(shape=(input_dim,)))
    for units in hidden_layers:
        model.add(layers.Dense(
            units, activation=activation,
            kernel_regularizer=regularizers.l2(l2)
        ))
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


# ---------------------------------------------------------------------
# 4. HYPERPARAMETER SEARCH (manual random search, GPU-trained each trial)
# ---------------------------------------------------------------------
PARAM_SPACE = {
    "hidden_layers": [
        (128, 64), (256, 128), (128, 64, 32), (256, 128, 64),
        (200, 100), (100, 50, 25), (256, 128, 64, 32), (150, 75),
    ],
    "dropout": [0.1, 0.2, 0.3, 0.4],
    "l2": [1e-6, 1e-5, 1e-4, 1e-3],
    "learning_rate": [0.0005, 0.001, 0.002, 0.003],
    "activation": ["relu", "elu"],
    "batch_size": [128, 256, 512],
}


def sample_params(rng):
    return {k: rng.choice(v) if not isinstance(v[0], tuple) else v[rng.integers(len(v))]
            for k, v in PARAM_SPACE.items()}


def tune_mlp(X_train, y_train, n_trials=15, cv_splits=3, max_epochs=60):
    rng = np.random.default_rng(RANDOM_STATE)
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)

    best_score, best_params = -1, None
    for trial in range(n_trials):
        params = sample_params(rng)
        fold_f1s = []

        for tr_idx, val_idx in cv.split(X_train, y_train):
            X_tr, X_val = X_train.iloc[tr_idx].values, X_train.iloc[val_idx].values
            y_tr, y_val = y_train.iloc[tr_idx].values, y_train.iloc[val_idx].values

            model = build_model(
                X_tr.shape[1], params["hidden_layers"], params["dropout"],
                params["l2"], params["learning_rate"], params["activation"]
            )
            early_stop = keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=6, restore_best_weights=True
            )
            model.fit(
                X_tr, y_tr, validation_data=(X_val, y_val),
                epochs=max_epochs, batch_size=params["batch_size"],
                callbacks=[early_stop], verbose=0,
            )
            y_pred = (model.predict(X_val, verbose=0).ravel() > 0.5).astype(int)
            fold_f1s.append(f1_score(y_val, y_pred))
            keras.backend.clear_session()

        mean_f1 = float(np.mean(fold_f1s))
        print(f"Trial {trial + 1}/{n_trials}: F1={mean_f1:.4f}  params={params}")
        if mean_f1 > best_score:
            best_score, best_params = mean_f1, params

    print(f"\nBest trial F1={best_score:.4f}\nBest params: {best_params}")
    return best_params


# ---------------------------------------------------------------------
# 5. SKLEARN-COMPATIBLE WRAPPER (lets us reuse permutation_importance,
#    confusion matrix / ROC plotting code unchanged)
# ---------------------------------------------------------------------
class KerasMLPWrapper:
    def __init__(self, params, max_epochs=100, patience=10):
        self.params = params
        self.max_epochs = max_epochs
        self.patience = patience
        self.model = None
        self.history_ = None

    def fit(self, X, y, X_val=None, y_val=None):
        X = np.asarray(X, dtype="float32")
        y = np.asarray(y, dtype="float32")
        self.model = build_model(
            X.shape[1], self.params["hidden_layers"], self.params["dropout"],
            self.params["l2"], self.params["learning_rate"], self.params["activation"]
        )
        early_stop = keras.callbacks.EarlyStopping(
            monitor="val_loss" if X_val is not None else "loss",
            patience=self.patience, restore_best_weights=True
        )
        validation_data = (X_val, y_val) if X_val is not None else None
        if validation_data is None:
            # carve out a small internal validation split for early stopping
            n_val = max(1, int(0.1 * len(X)))
            X_val_, y_val_ = X[-n_val:], y[-n_val:]
            X_tr_, y_tr_ = X[:-n_val], y[:-n_val]
            validation_data = (X_val_, y_val_)
        else:
            X_tr_, y_tr_ = X, y

        hist = self.model.fit(
            X_tr_, y_tr_, validation_data=validation_data,
            epochs=self.max_epochs, batch_size=self.params["batch_size"],
            callbacks=[early_stop], verbose=0,
        )
        self.history_ = hist.history
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype="float32")
        p = self.model.predict(X, verbose=0).ravel()
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.5).astype(int)

    def score(self, X, y):
        # required by sklearn's permutation_importance when no scoring fn given
        return f1_score(y, self.predict(X))


# ---------------------------------------------------------------------
# 6. EVALUATION (5-fold stratified CV, same protocol as before)
# ---------------------------------------------------------------------
def evaluate(best_params, X, y, cv_splits=5, max_epochs=100):
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
    accs, precs, recs, f1s, aucs = [], [], [], [], []

    for train_idx, test_idx in cv.split(X, y):
        X_tr, X_te = X.iloc[train_idx].values, X.iloc[test_idx].values
        y_tr, y_te = y.iloc[train_idx].values, y.iloc[test_idx].values

        wrapper = KerasMLPWrapper(best_params, max_epochs=max_epochs)
        wrapper.fit(X_tr, y_tr)

        y_pred = wrapper.predict(X_te)
        y_proba = wrapper.predict_proba(X_te)[:, 1]

        accs.append(accuracy_score(y_te, y_pred))
        precs.append(precision_score(y_te, y_pred))
        recs.append(recall_score(y_te, y_pred))
        f1s.append(f1_score(y_te, y_pred))
        aucs.append(roc_auc_score(y_te, y_proba))
        keras.backend.clear_session()

    return {
        "Accuracy": (np.mean(accs), np.std(accs)),
        "Precision": (np.mean(precs), np.std(precs)),
        "Recall": (np.mean(recs), np.std(recs)),
        "F1-Score": (np.mean(f1s), np.std(f1s)),
        "AUC": (np.mean(aucs), np.std(aucs)),
    }


# ---------------------------------------------------------------------
# 7. COMPARISON TABLE
# ---------------------------------------------------------------------
REFERENCE_RESULTS = {
    "Li (2026) - MLP (UNSW-NB15 binary)": {
        "Accuracy": 0.8578, "Precision": 0.8432, "Recall": 0.8896,
        "F1-Score": 0.8498, "AUC": np.nan,
    },
    "Mohale & Obagbuwa (2025) - MLP Classifier": {
        "Accuracy": 0.8598, "Precision": 0.8400, "Recall": 0.8700,
        "F1-Score": 0.8500, "AUC": 0.9100,
    },
}


def build_comparison_table(my_results, out_csv=os.path.join(OUT_DIR, "comparison_table.csv")):
    rows = []
    for name, metrics in REFERENCE_RESULTS.items():
        row = {"Model": name}
        row.update(metrics)
        rows.append(row)

    my_row = {"Model": "Proposed Tuned MLP -- GPU (ALL features, no selection)"}
    for k, (mean, _std) in my_results.items():
        my_row[k] = round(mean, 4)
    rows.append(my_row)

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(df.to_string(index=False))

    target_acc = max(REFERENCE_RESULTS[k]["Accuracy"] for k in REFERENCE_RESULTS)
    achieved_acc = my_results["Accuracy"][0]
    print(f"\nTarget accuracy to beat (both papers): {target_acc:.4f}")
    print(f"Achieved accuracy (5-fold CV mean):     {achieved_acc:.4f}")
    print("RESULT:", "PASS - beats both papers" if achieved_acc > target_acc
          else "FAIL - does not beat both papers, consider more trials/epochs")

    return df


# ---------------------------------------------------------------------
# 8. GRAPHS
# ---------------------------------------------------------------------
def plot_comparison_bar(df, out_path=os.path.join(OUT_DIR, "comparison_bar.png")):
    metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]
    x = np.arange(len(df))
    width = 0.2
    fig, ax = plt.subplots(figsize=(10, 6))
    for i, m in enumerate(metrics):
        ax.bar(x + i * width, df[m], width, label=m)
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(df["Model"], rotation=15, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title("MLP (GPU) -- Proposed Model vs. Reference Papers (UNSW-NB15 binary)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_confusion_matrix(wrapper, X_test, y_test, out_path=os.path.join(OUT_DIR, "confusion_matrix.png")):
    y_pred = wrapper.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Malicious"])
    fig, ax = plt.subplots(figsize=(5, 5))
    disp.plot(ax=ax, cmap="Blues", values_format="d")
    ax.set_title("Confusion Matrix -- Tuned MLP (GPU)")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_roc(wrapper, X_test, y_test, out_path=os.path.join(OUT_DIR, "roc_curve.png")):
    y_proba = wrapper.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f"Tuned MLP (AUC = {auc:.3f})")
    plt.plot([0, 1], [0, 1], "--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve -- Tuned MLP (GPU, All Features)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_loss_curve(wrapper, out_path=os.path.join(OUT_DIR, "loss_curve.png")):
    if wrapper.history_ is None:
        return
    plt.figure(figsize=(7, 5))
    plt.plot(wrapper.history_["loss"], label="Training loss")
    if "val_loss" in wrapper.history_:
        plt.plot(wrapper.history_["val_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("MLP Training Loss Curve (GPU)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_permutation_importance(wrapper, X_test, y_test, feature_names,
                                 out_path=os.path.join(OUT_DIR, "permutation_importance.png"),
                                 top_n=15):
    result = permutation_importance(
        wrapper, X_test.values, y_test.values, n_repeats=5,
        random_state=RANDOM_STATE, n_jobs=1  # keep n_jobs=1 -- GPU models don't parallelize across processes well
    )
    order = np.argsort(result.importances_mean)[::-1][:top_n]
    plt.figure(figsize=(9, max(6, 0.35 * top_n)))
    plt.barh(
        [feature_names[i] for i in order][::-1],
        result.importances_mean[order][::-1],
        xerr=result.importances_std[order][::-1],
        color="darkorange",
    )
    plt.xlabel("Mean F1 decrease when permuted")
    plt.title(f"MLP Permutation Importance -- top {top_n} of {len(feature_names)} features")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


# ---------------------------------------------------------------------
# 9. MAIN
# ---------------------------------------------------------------------
if __name__ == "__main__":
    configure_gpu()

    # >>>> EDIT THESE PATHS to point at your local UNSW-NB15 files <<<<
    TRAIN_CSV = "UNSW_NB15_training-set.csv"
    TEST_CSV = "UNSW_NB15_testing-set.csv"

    print("Loading data...")
    df = load_unsw_nb15(train_path=TRAIN_CSV, test_path=TEST_CSV)

    print("Preprocessing (no feature selection -- all columns kept)...")
    X, y = preprocess(df)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )

    print("\nHyperparameter search (random search, each trial trained on GPU)...")
    t0 = time.time()
    best_params = tune_mlp(X_train, y_train, n_trials=15, cv_splits=3, max_epochs=60)
    print(f"Search finished in {time.time() - t0:.1f}s")

    print("\n5-fold CV evaluation of the tuned model on the FULL dataset...")
    cv_results = evaluate(best_params, X, y, cv_splits=5, max_epochs=100)
    for k, (m, s) in cv_results.items():
        print(f"{k}: {m:.4f} +/- {s:.4f}")

    print("\nBuilding comparison table against the 2 reference papers...")
    comp_df = build_comparison_table(cv_results)

    print("\nGenerating graphs (final fit on the train/test split)...")
    final_wrapper = KerasMLPWrapper(best_params, max_epochs=100)
    final_wrapper.fit(X_train.values, y_train.values, X_test.values, y_test.values)

    plot_comparison_bar(comp_df)
    plot_confusion_matrix(final_wrapper, X_test, y_test)
    plot_roc(final_wrapper, X_test, y_test)
    plot_loss_curve(final_wrapper)
    plot_permutation_importance(final_wrapper, X_test, y_test, X.columns.tolist())

    print(f"\nAll outputs saved to ./{OUT_DIR}/")

# Logistic Regression

In [ ]:
"""
Logistic Regression (Binary Classification) on UNSW-NB15 -- FULL FEATURE SET
========================================================================
Same preprocessing pipeline as the XGBoost/CatBoost companion scripts
(NO feature selection -- all original columns kept; encoders/scaler fit
on train only, applied to test). This script targets the two papers
that actually report a Logistic Regression number on UNSW-NB15:

  [1] Hakke, D.G. et al. (2025). Int. J. Applied Mathematics, 38(3s), 447.
      -> Logistic Regression: Accuracy=88.69%, Precision(Attack)=0.85,
         Recall(Attack)=0.99, F1(Attack)=0.91

  [2] Kasongo, S.M. & Sun, Y. (2020). J Big Data, 7:105.
      -> Logistic Regression (42 features, no feature selection --
         their own reduced 19-feature run actually scored LOWER at
         77.64%, so the 42-feature number is both the harder target
         and the fairer one to compare against a no-selection pipeline):
         Test Acc=79.59%, Precision=73.32%, Recall=98.94%, F1=84.22%

WHY THIS SCRIPT LOOKS DIFFERENT FROM THE XGBOOST/CATBOOST VERSIONS:
  1. No GPU section. sklearn's LogisticRegression is CPU-only regardless
     of hardware -- there's nothing to detect or configure.
  2. No search-subsample trick. LR trains in well under a second even
     on the full ~200k-row training set with 'saga', so the "search on
     50k rows, then refit on everything" optimization the tree scripts
     need for speed is unnecessary overhead here -- the search runs
     directly on the full training data.
  3. Regularization search replaces depth/estimator search: penalty
     type (L1 / L2 / elasticnet), C (inverse regularization strength),
     and class_weight are the parameters that actually move a linear
     model's performance on this data.
  4. Decision-threshold tuning is added as an explicit extra stage.
     Both reference papers report metrics at the default 0.5 threshold.
     On a 63.9%-attack dataset, 0.5 is not necessarily where F1 is
     maximized. We tune the threshold on a validation carve-out from
     TRAINING data only (never on the test fold), so this is a
     legitimate decision-rule refinement, not test-set leakage.

HOW TO USE
----------
1. Get UNSW_NB15_training-set.csv / UNSW_NB15_testing-set.csv locally.
2. Edit the paths in the __main__ block if needed.
3. Run: python logreg_unsw_nb15_full_features.py
4. Outputs land in ./outputs/ (kept under *_logreg.* filenames).
"""

import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
)
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)


# ---------------------------------------------------------------------
# 1. LOAD DATA  (identical to the XGBoost / CatBoost scripts)
# ---------------------------------------------------------------------
def load_unsw_nb15(train_path=None, test_path=None):
    df_train = pd.read_csv(
        "/kaggle/input/datasets/ajeetkumar20/unsw-nb15-v1-1/UNSW_NB15_training-set.csv"
    )
    df_test = pd.read_csv(
        "/kaggle/input/datasets/ajeetkumar20/unsw-nb15-v1-1/UNSW_NB15_testing-set.csv"
    )
    return df_train, df_test


# ---------------------------------------------------------------------
# 2. PREPROCESSING  (NO FEATURE SELECTION -- byte-for-byte identical to
#    the XGBoost/CatBoost scripts, so results are directly comparable)
# ---------------------------------------------------------------------
def preprocess_train_test(df_train, df_test):
    """
    Preprocess train and test separately.
    Encoders and scaler are FIT ONLY on training data.
    """
    train = df_train.copy()
    test = df_test.copy()

    for id_col in ["id", "ID", "Id"]:
        if id_col in train.columns:
            train.drop(columns=[id_col], inplace=True)
        if id_col in test.columns:
            test.drop(columns=[id_col], inplace=True)

    y_train = train["label"].astype(int)
    y_test = test["label"].astype(int)
    train.drop(columns=["label"], inplace=True)
    test.drop(columns=["label"], inplace=True)

    if "attack_cat" in train.columns:
        train.drop(columns=["attack_cat"], inplace=True)
    if "attack_cat" in test.columns:
        test.drop(columns=["attack_cat"], inplace=True)

    encoders = {}
    categorical_cols = train.select_dtypes(include=["object"]).columns
    for col in categorical_cols:
        le = LabelEncoder()
        combined = pd.concat([
            train[col].astype(str),
            test[col].astype(str)
        ])
        le.fit(combined)
        train[col] = le.transform(train[col].astype(str))
        test[col] = le.transform(test[col].astype(str))
        encoders[col] = le

    scaler = MinMaxScaler()
    X_train = pd.DataFrame(
        scaler.fit_transform(train),
        columns=train.columns
    )
    X_test = pd.DataFrame(
        scaler.transform(test),
        columns=test.columns
    )

    print(f"Training features : {X_train.shape}")
    print(f"Testing features  : {X_test.shape}")

    return X_train, X_test, y_train, y_test, scaler, encoders


# ---------------------------------------------------------------------
# 3. THRESHOLD OPTIMIZATION HELPER
# ---------------------------------------------------------------------
def optimize_threshold(y_true, y_proba, metric="f1"):
    """
    Scans thresholds 0.01-0.99 and returns the one that maximizes F1
    (or another metric) on the given (validation-only) labels/probas.
    NEVER call this with test-set labels -- it must only see data the
    final evaluation won't be scored on, or the reported metric leaks.
    """
    best_thr, best_score = 0.5, -1.0
    for thr in np.arange(0.01, 1.00, 0.01):
        y_pred = (y_proba >= thr).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0) if metric == "f1" \
            else accuracy_score(y_true, y_pred)
        if score > best_score:
            best_score, best_thr = score, thr
    return best_thr, best_score


# ---------------------------------------------------------------------
# 4. HYPERPARAMETER TUNING (2-stage: RandomizedSearch -> GridSearch;
#    no subsampling needed -- LR is fast enough to search on full data)
# ---------------------------------------------------------------------
def tune_logreg(X_train, y_train):
    base_lr = LogisticRegression(
        solver="saga",       # supports l1, l2, elasticnet -- and scales to this data size
        max_iter=3000,       # MinMax-scaled features can need more iterations than StandardScaler
        tol=1e-4,
        random_state=RANDOM_STATE,
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    # ---- Stage 1: broad RandomizedSearchCV ----
    # Split by penalty type since l1_ratio only applies to elasticnet,
    # and liblinear/lbfgs don't support every penalty -- using a list of
    # dicts avoids invalid combinations rather than filtering afterward.
    param_dist = [
        {
            "penalty": ["l1"],
            "C": [0.001, 0.01, 0.1, 0.5, 1, 5, 10, 50, 100],
            "class_weight": [None, "balanced"],
        },
        {
            "penalty": ["l2"],
            "C": [0.001, 0.01, 0.1, 0.5, 1, 5, 10, 50, 100],
            "class_weight": [None, "balanced"],
        },
        {
            "penalty": ["elasticnet"],
            "C": [0.001, 0.01, 0.1, 0.5, 1, 5, 10, 50, 100],
            "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9],
            "class_weight": [None, "balanced"],
        },
    ]

    t0 = time.time()
    rand_search = RandomizedSearchCV(
        base_lr, param_distributions=param_dist, n_iter=40,
        scoring="f1", cv=cv, n_jobs=-1, random_state=RANDOM_STATE, verbose=1
    )
    rand_search.fit(X_train, y_train)
    best_rand = rand_search.best_params_
    print(f"Stage-1 (Randomized) best params: {best_rand}")
    print(f"Stage-1 best CV F1: {rand_search.best_score_:.4f}")
    print(f"Stage-1 took {time.time() - t0:.1f}s")

    # ---- Stage 2: narrow GridSearchCV around the best region ----
    def widen_log(v, factor=3):
        return sorted({round(v / factor, 5), v, round(v * factor, 5)})

    penalty = best_rand["penalty"]
    grid = {
        "penalty": [penalty],
        "C": widen_log(best_rand["C"]),
        "class_weight": [best_rand["class_weight"]],
    }
    if penalty == "elasticnet":
        v = best_rand["l1_ratio"]
        grid["l1_ratio"] = sorted({max(0.0, v - 0.15), v, min(1.0, v + 0.15)})

    t0 = time.time()
    grid_search = GridSearchCV(
        base_lr, param_grid=grid, scoring="f1", cv=cv, n_jobs=-1, verbose=1
    )
    grid_search.fit(X_train, y_train)
    best_params = grid_search.best_params_
    print(f"Stage-2 (Grid) best params: {best_params}")
    print(f"Stage-2 best CV F1: {grid_search.best_score_:.4f}")
    print(f"Stage-2 took {time.time() - t0:.1f}s")

    final_model = LogisticRegression(
        solver="saga", max_iter=3000, tol=1e-4,
        random_state=RANDOM_STATE, **best_params,
    )
    print(f"\nFinal tuned hyperparameters: {best_params}")
    return final_model, best_params


# ---------------------------------------------------------------------
# 5. EVALUATION
#    Two protocols are reported:
#    (a) 5-fold stratified CV on the full combined dataset (train+test
#        recombined) -- matches the CV protocol used in the XGBoost/
#        CatBoost companion scripts, our most statistically robust number.
#    (b) The OFFICIAL UNSW-NB15 train/test split -- matches what both
#        reference papers actually evaluate on (a single held-out test
#        set, not CV), for an apples-to-apples comparison.
#    Both are reported at the default 0.5 threshold AND at a threshold
#    tuned on a validation carve-out from training data only.
# ---------------------------------------------------------------------
def _score(y_true, y_pred, y_proba):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0),
        "AUC": roc_auc_score(y_true, y_proba),
    }


def evaluate_cv(model, X, y, cv_splits=5):
    """5-fold CV. For each fold, the threshold is tuned on a further
    carve-out of that fold's TRAINING split only -- the test fold never
    influences threshold selection."""
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
    default_runs, tuned_runs, thresholds = [], [], []
    t0 = time.time()

    for fold_i, (train_idx, test_idx) in enumerate(cv.split(X, y), 1):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        X_fit, X_thr, y_fit, y_thr = train_test_split(
            X_tr, y_tr, test_size=0.15, stratify=y_tr, random_state=RANDOM_STATE
        )

        model.fit(X_fit, y_fit)
        y_proba_thr = model.predict_proba(X_thr)[:, 1]
        best_thr, _ = optimize_threshold(y_thr, y_proba_thr)
        thresholds.append(best_thr)

        y_proba_te = model.predict_proba(X_te)[:, 1]
        default_runs.append(_score(y_te, (y_proba_te >= 0.5).astype(int), y_proba_te))
        tuned_runs.append(_score(y_te, (y_proba_te >= best_thr).astype(int), y_proba_te))

        print(f"  Fold {fold_i}/{cv_splits} done "
              f"(tuned threshold={best_thr:.2f}, {time.time() - t0:.1f}s elapsed)")

    def summarize(runs):
        keys = runs[0].keys()
        return {k: (np.mean([r[k] for r in runs]), np.std([r[k] for r in runs])) for k in keys}

    return summarize(default_runs), summarize(tuned_runs), np.mean(thresholds)


def evaluate_official_split(model, X_train, X_test, y_train, y_test):
    """Fit on the official training set, tune the threshold on a
    validation carve-out from training data, evaluate once on the
    official test set -- mirrors the papers' own protocol."""
    X_fit, X_thr, y_fit, y_thr = train_test_split(
        X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE
    )
    model.fit(X_fit, y_fit)

    y_proba_thr = model.predict_proba(X_thr)[:, 1]
    best_thr, _ = optimize_threshold(y_thr, y_proba_thr)

    y_proba_te = model.predict_proba(X_test)[:, 1]
    default_result = _score(y_test, (y_proba_te >= 0.5).astype(int), y_proba_te)
    tuned_result = _score(y_test, (y_proba_te >= best_thr).astype(int), y_proba_te)

    # Refit on ALL official training data (not just the 85% subtrain)
    # for the final model used in the plots below, keeping the tuned threshold.
    model.fit(X_train, y_train)
    return default_result, tuned_result, best_thr, model


# ---------------------------------------------------------------------
# 6. COMPARISON TABLE (against the 2 reference papers' LR numbers)
# ---------------------------------------------------------------------
REFERENCE_RESULTS = {
    "Hakke et al. (2025) - LR": {
        "Accuracy": 0.8869, "Precision": 0.85, "Recall": 0.99,
        "F1-Score": 0.91, "AUC": np.nan,
    },
    "Kasongo & Sun (2020) - LR (42 features, no selection)": {
        "Accuracy": 0.7959, "Precision": 0.7332, "Recall": 0.9894,
        "F1-Score": 0.8422, "AUC": np.nan,
    },
}


def build_comparison_table(cv_default, cv_tuned, split_default, split_tuned,
                            out_csv=os.path.join(OUT_DIR, "comparison_table_logreg.csv")):
    rows = []
    for name, metrics in REFERENCE_RESULTS.items():
        row = {"Model": name}
        row.update(metrics)
        rows.append(row)

    def add_row(name, metrics_dict, is_cv):
        row = {"Model": name}
        for k, v in metrics_dict.items():
            row[k] = round(v[0], 4) if is_cv else round(v, 4)
        rows.append(row)

    add_row("Proposed LR -- 5-fold CV, default thr=0.50", cv_default, is_cv=True)
    add_row("Proposed LR -- 5-fold CV, tuned threshold", cv_tuned, is_cv=True)
    add_row("Proposed LR -- official split, default thr=0.50", split_default, is_cv=False)
    add_row("Proposed LR -- official split, tuned threshold", split_tuned, is_cv=False)

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(df.to_string(index=False))
    return df


# ---------------------------------------------------------------------
# 7. GRAPHS
# ---------------------------------------------------------------------
def plot_comparison_bar(df, out_path=os.path.join(OUT_DIR, "comparison_bar_logreg.png")):
    metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]
    x = np.arange(len(df))
    width = 0.2

    fig, ax = plt.subplots(figsize=(13, 6))
    for i, m in enumerate(metrics):
        ax.bar(x + i * width, df[m], width, label=m)
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(df["Model"], rotation=20, ha="right", fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title("Logistic Regression -- Proposed Model vs. Reference Papers")
    ax.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_confusion_matrix(model, X_test, y_test, threshold,
                           out_path=os.path.join(OUT_DIR, "confusion_matrix_logreg.png")):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Malicious"])
    fig, ax = plt.subplots(figsize=(5, 5))
    disp.plot(ax=ax, cmap="Blues", values_format="d")
    ax.set_title(f"Confusion Matrix -- Tuned LR (threshold={threshold:.2f})")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_roc(model, X_test, y_test, out_path=os.path.join(OUT_DIR, "roc_curve_logreg.png")):
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)

    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, color="steelblue", label=f"Tuned LR (AUC = {auc:.3f})")
    plt.plot([0, 1], [0, 1], "--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve -- Tuned Logistic Regression (All Features)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_threshold_curve(model, X_thr, y_thr, best_thr,
                          out_path=os.path.join(OUT_DIR, "threshold_curve_logreg.png")):
    y_proba = model.predict_proba(X_thr)[:, 1]
    thresholds = np.arange(0.01, 1.00, 0.01)
    f1s = [f1_score(y_thr, (y_proba >= t).astype(int), zero_division=0) for t in thresholds]

    plt.figure(figsize=(7, 5))
    plt.plot(thresholds, f1s, color="darkorange")
    plt.axvline(best_thr, color="gray", linestyle="--", label=f"Chosen threshold = {best_thr:.2f}")
    plt.axvline(0.5, color="lightgray", linestyle=":", label="Default threshold = 0.50")
    plt.xlabel("Decision Threshold")
    plt.ylabel("F1-Score (Attack class)")
    plt.title("Threshold Tuning Curve (on validation carve-out, not test data)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_coefficients(model, feature_names, out_path=os.path.join(OUT_DIR, "coefficients_logreg.png")):
    coefs = model.coef_[0]
    order = np.argsort(np.abs(coefs))[::-1]

    plt.figure(figsize=(9, max(6, 0.28 * len(order))))
    colors = ["crimson" if coefs[i] < 0 else "steelblue" for i in order]
    plt.barh([feature_names[i] for i in order][::-1], coefs[order][::-1], color=colors[::-1])
    plt.xlabel("Coefficient (on MinMax-scaled features -- directly comparable in magnitude)")
    plt.title("Logistic Regression Coefficients -- blue pushes toward 'attack', red toward 'normal'")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


# ---------------------------------------------------------------------
# 8. MAIN
# ---------------------------------------------------------------------
if __name__ == "__main__":
    TRAIN_CSV = "UNSW_NB15_training-set.csv"
    TEST_CSV = "UNSW_NB15_testing-set.csv"

    pipeline_start = time.time()

    print("Loading data...")
    df_train, df_test = load_unsw_nb15(train_path=TRAIN_CSV, test_path=TEST_CSV)

    print("Preprocessing (no feature selection -- all columns kept, "
          "encoders/scaler fit on train only)...")
    X_train, X_test, y_train, y_test, scaler, encoders = preprocess_train_test(df_train, df_test)

    X_full = pd.concat([X_train, X_test], axis=0).reset_index(drop=True)
    y_full = pd.concat([y_train, y_test], axis=0).reset_index(drop=True)

    print("\nHyperparameter tuning (Random -> Grid, on full training data)...")
    t0 = time.time()
    best_model, best_params = tune_logreg(X_train, y_train)
    print(f"\nTotal tuning time: {time.time() - t0:.1f}s")

    print("\n5-fold CV evaluation (default vs. tuned threshold) on the FULL dataset...")
    cv_default, cv_tuned, mean_thr = evaluate_cv(best_model, X_full, y_full, cv_splits=5)
    print(f"Mean tuned threshold across folds: {mean_thr:.3f}")
    for label, results in [("Default (0.50)", cv_default), ("Tuned", cv_tuned)]:
        print(f"\n-- CV, {label} threshold --")
        for k, (m, s) in results.items():
            print(f"{k}: {m:.4f} +/- {s:.4f}")

    print("\nEvaluating on the OFFICIAL train/test split (matches the papers' own protocol)...")
    split_default, split_tuned, best_thr, fitted_model = evaluate_official_split(
        best_model, X_train, X_test, y_train, y_test
    )
    print(f"Threshold tuned on official split: {best_thr:.3f}")
    for label, results in [("Default (0.50)", split_default), ("Tuned", split_tuned)]:
        print(f"\n-- Official split, {label} threshold --")
        for k, v in results.items():
            print(f"{k}: {v:.4f}")

    print("\nBuilding comparison table against the 2 reference papers...")
    comp_df = build_comparison_table(cv_default, cv_tuned, split_default, split_tuned)

    print("\nGenerating graphs...")
    # Validation carve-out for the threshold curve plot (re-derive it here
    # so the plot matches exactly what evaluate_official_split used internally)
    X_fit, X_thr_plot, y_fit, y_thr_plot = train_test_split(
        X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE
    )
    plot_comparison_bar(comp_df)
    plot_confusion_matrix(fitted_model, X_test, y_test, best_thr)
    plot_roc(fitted_model, X_test, y_test)
    plot_threshold_curve(fitted_model, X_thr_plot, y_thr_plot, best_thr)
    plot_coefficients(fitted_model, X_full.columns.tolist())

    print(f"\nAll outputs saved to ./{OUT_DIR}/ (with *_logreg suffix)")
    print(f"\nTOTAL PIPELINE TIME: {time.time() - pipeline_start:.1f}s")

# catboost new implementation 

In [ ]:
"""
CatBoost (Binary Classification) on UNSW-NB15 -- FULL FEATURE SET, FAST VERSION
========================================================================
Same preprocessing pipeline as before (NO feature selection). This version
fixes a well-documented CatBoost behavior: GPU training has a large FIXED
per-fit overhead (CatBoost GitHub issues #2084, #1034, #629 all report
~60s overhead per fit regardless of data size, and hangs after many
sequential fits) -- meaning GPU is actively counterproductive for
hyperparameter search (many small fits) and should only be used for the
few large, long-running fits.

WHAT CHANGED FROM THE PREVIOUS VERSION
---------------------------------------
  1. Hyperparameter search (Stages 1-2) now runs on CPU with a manual
     search loop (not sklearn's *SearchCV), using all CPU cores via
     thread_count=-1. This is what was silently taking hours.
  2. GPU (if detected) is used ONLY for Stage 3 (early-stopping
     refinement on the full training set) and Stage 4 (final 5-fold CV)
     -- a handful of large fits, which is what GPU actually accelerates.
  3. tqdm progress bars on every loop: Stage 1, Stage 2, and the final
     5-fold CV, so you always see live progress instead of a silent gap.
  4. A hard wall-clock TIME BUDGET on Stages 1 and 2 (25 min each by
     default) -- if the search hasn't finished by then, it stops and
     keeps the best candidate found so far, guaranteeing the whole
     pipeline can't silently run for hours.
  5. Trimmed candidate counts (still a real random-then-grid search,
     just sized to comfortably fit a 1-2 hour total budget).

Expected total runtime: roughly 35-75 minutes on a typical Kaggle
GPU session (T4), safely inside a 1-2 hour ceiling even if some stage
runs slower than expected -- printed timings at the end of every stage
let you see exactly where the time goes.

Reference results (same 3 papers, same goal: beat the best reported
figure in each -- Kasongo & Sun report no standalone XGBoost number,
so they're marked N/A):

  [1] Alkhater, N. (2026). Computers, 15(5), 285.
      -> XGBoost (default config, 5-fold CV, 48 features):
         Acc=0.97+-0.005, Prec=0.96+-0.006, Recall=0.96+-0.005,
         F1=0.96+-0.006, AUC=0.980+-0.003

  [2] Kasongo, S.M. & Sun, Y. (2020). J Big Data, 7:105.
      -> XGBoost used only for feature-importance/selection -- N/A.

  [3] Mohale, V.Z. & Obagbuwa, I.C. (2025). Front. Comput. Sci., 7:1520741.
      -> XGBoost: Acc=86.87%, Precision=0.85, Recall=0.88, F1=0.86,
         ROC-AUC=0.93, FPR=0.08, FNR=0.11

Requires: pip install catboost tqdm   (use --break-system-packages if needed)
"""

import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
)

from catboost import CatBoostClassifier, CatBoostError
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

SEARCH_SAMPLE_SIZE = 50_000       # rows used ONLY for hyperparameter search
STAGE_TIME_BUDGET_S = 25 * 60     # hard cap per search stage: 25 minutes


# ---------------------------------------------------------------------
# 0. GPU DETECTION (used only for Stages 3-4, the large fits)
# ---------------------------------------------------------------------
def detect_and_configure_gpu():
    dummy_X = np.random.rand(200, 5)
    dummy_y = np.random.randint(0, 2, 200)
    try:
        probe = CatBoostClassifier(task_type="GPU", devices="0", iterations=5, verbose=False)
        probe.fit(dummy_X, dummy_y)
        print("=" * 60)
        print("GPU DETECTED -- Stages 3-4 (the large fits) will use it.")
        print("Stages 1-2 (hyperparameter search) use CPU regardless --")
        print("GPU has ~60s fixed overhead per fit, which makes it SLOWER")
        print("for many small search fits (see catboost issue #2084).")
        print("=" * 60)
        return {"task_type": "GPU", "devices": "0"}
    except CatBoostError as e:
        print("=" * 60)
        print("GPU NOT DETECTED -- Stages 3-4 will also run on CPU.")
        print(f"(CatBoost reported: {str(e)[:120]})")
        print("If you're on Kaggle: Settings -> Accelerator -> GPU T4 x2")
        print("=" * 60)
        return {"task_type": "CPU", "thread_count": -1}


# ---------------------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------------------
def load_unsw_nb15(train_path=None, test_path=None):
    df_train = pd.read_csv("/kaggle/input/datasets/ajeetkumar20/unsw-nb15-v1-1/UNSW_NB15_training-set.csv")
    df_test = pd.read_csv("/kaggle/input/datasets/ajeetkumar20/unsw-nb15-v1-1/UNSW_NB15_testing-set.csv")
    return df_train, df_test


# ---------------------------------------------------------------------
# 2. PREPROCESSING (unchanged -- no feature selection, encoders/scaler
#    fit on train only)
# ---------------------------------------------------------------------
def preprocess_train_test(df_train, df_test):
    train = df_train.copy()
    test = df_test.copy()

    for id_col in ["id", "ID", "Id"]:
        if id_col in train.columns:
            train.drop(columns=[id_col], inplace=True)
        if id_col in test.columns:
            test.drop(columns=[id_col], inplace=True)

    y_train = train["label"].astype(int)
    y_test = test["label"].astype(int)
    train.drop(columns=["label"], inplace=True)
    test.drop(columns=["label"], inplace=True)

    if "attack_cat" in train.columns:
        train.drop(columns=["attack_cat"], inplace=True)
    if "attack_cat" in test.columns:
        test.drop(columns=["attack_cat"], inplace=True)

    encoders = {}
    categorical_cols = train.select_dtypes(include=["object"]).columns
    for col in categorical_cols:
        le = LabelEncoder()
        combined = pd.concat([train[col].astype(str), test[col].astype(str)])
        le.fit(combined)
        train[col] = le.transform(train[col].astype(str))
        test[col] = le.transform(test[col].astype(str))
        encoders[col] = le

    scaler = MinMaxScaler()
    X_train = pd.DataFrame(scaler.fit_transform(train), columns=train.columns)
    X_test = pd.DataFrame(scaler.transform(test), columns=test.columns)

    print(f"Training features : {X_train.shape}")
    print(f"Testing features  : {X_test.shape}")

    return X_train, X_test, y_train, y_test, scaler, encoders


# ---------------------------------------------------------------------
# 3. HYPERPARAMETER TUNING -- manual search loop, CPU-only, tqdm +
#    hard time budget, then GPU-or-fallback for the two large fits.
# ---------------------------------------------------------------------
def make_search_subsample(X_train, y_train, sample_size=SEARCH_SAMPLE_SIZE, random_state=RANDOM_STATE):
    if len(X_train) <= sample_size:
        return X_train, y_train
    X_search, _, y_search, _ = train_test_split(
        X_train, y_train, train_size=sample_size, stratify=y_train, random_state=random_state
    )
    print(f"Hyperparameter search will use a stratified subsample: "
          f"{len(X_search)} / {len(X_train)} rows ({100 * len(X_search) / len(X_train):.1f}%)")
    return X_search, y_search


def run_candidate_search(X, y, candidates, cv_folds, base_params, time_budget_s, desc):
    """Evaluate a list of hyperparameter-dict candidates via k-fold CV,
    with a live tqdm progress bar and a hard wall-clock time budget --
    if the budget is hit mid-search, keep whatever's been found so far
    instead of running unbounded."""
    skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=RANDOM_STATE)
    results = []
    t_start = time.time()
    total_fits = len(candidates) * cv_folds

    pbar = tqdm(total=total_fits, desc=desc)
    stopped_early = False
    for cand in candidates:
        if time.time() - t_start > time_budget_s:
            stopped_early = True
            break
        fold_scores = []
        for train_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            model = CatBoostClassifier(**base_params, **cand, verbose=False)
            model.fit(X_tr, y_tr)
            preds = model.predict(X_val)
            fold_scores.append(f1_score(y_val, preds))
            pbar.update(1)
        results.append({"params": cand, "mean_f1": float(np.mean(fold_scores))})
    pbar.close()

    if stopped_early:
        print(f"[{desc}] Time budget ({time_budget_s / 60:.0f} min) reached -- "
              f"stopped after {len(results)}/{len(candidates)} candidates. "
              f"Using the best one found so far.")

    results.sort(key=lambda r: r["mean_f1"], reverse=True)
    print(f"[{desc}] Best: {results[0]['params']}  (CV F1 = {results[0]['mean_f1']:.4f})")
    return results[0]["params"], results[0]["mean_f1"]


def widen(v, lo_bound, step, as_int=True):
    vals = {v - step, v, v + step}
    vals = {max(lo_bound, x) for x in vals}
    return sorted({int(x) if as_int else round(x, 4) for x in vals})


def tune_catboost(X_train, y_train, gpu_params):
    X_search, y_search = make_search_subsample(X_train, y_train)

    cpu_search_params = {
        "task_type": "CPU", "thread_count": -1,
        "loss_function": "Logloss", "eval_metric": "Logloss",
        "random_seed": RANDOM_STATE,
    }

    # ---- Stage 1: random search over the full space, CPU, 3-fold ----
    param_space = {
        "iterations": [150, 250, 400],
        "depth": [4, 5, 6, 7, 8, 9, 10],
        "learning_rate": [0.02, 0.03, 0.05, 0.08, 0.1, 0.15, 0.2],
        "l2_leaf_reg": [1, 3, 5, 7, 9, 12, 15],
        "border_count": [32, 64, 128, 254],
        "bagging_temperature": [0, 0.2, 0.5, 1.0, 2.0],
        "random_strength": [0, 0.5, 1, 2, 5],
    }
    rng = np.random.RandomState(RANDOM_STATE)
    n_random_candidates = 25
    candidates = [
        {k: rng.choice(v).item() if hasattr(rng.choice(v), "item") else rng.choice(v)
         for k, v in param_space.items()}
        for _ in range(n_random_candidates)
    ]

    t0 = time.time()
    best_rand, _ = run_candidate_search(
        X_search, y_search, candidates, cv_folds=3, base_params=cpu_search_params,
        time_budget_s=STAGE_TIME_BUDGET_S, desc="Stage 1: Random search (CPU)"
    )
    print(f"Stage-1 wall time: {(time.time() - t0) / 60:.1f} min")

    # ---- Stage 2: narrow grid around the Stage-1 winner, CPU, 3-fold ----
    grid_candidates = []
    for it in widen(best_rand["iterations"], 100, 100):
        for d in widen(best_rand["depth"], 2, 1):
            for lr in widen(best_rand["learning_rate"], 0.02, 0.02, as_int=False):
                grid_candidates.append({
                    "iterations": it, "depth": d, "learning_rate": lr,
                    "l2_leaf_reg": best_rand["l2_leaf_reg"],
                    "border_count": best_rand["border_count"],
                    "bagging_temperature": best_rand["bagging_temperature"],
                    "random_strength": best_rand["random_strength"],
                })

    t0 = time.time()
    best_params, _ = run_candidate_search(
        X_search, y_search, grid_candidates, cv_folds=3, base_params=cpu_search_params,
        time_budget_s=STAGE_TIME_BUDGET_S, desc="Stage 2: Grid refine (CPU)"
    )
    print(f"Stage-2 wall time: {(time.time() - t0) / 60:.1f} min")

    # ---- Stage 3: early-stopping refinement of "iterations" on the FULL
    #      training set, using GPU if available (this is the first fit
    #      large/long enough for GPU to actually help). ----
    t0 = time.time()
    X_fit, X_es_val, y_fit, y_es_val = train_test_split(
        X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE
    )
    es_params = {k: v for k, v in best_params.items() if k != "iterations"}
    es_model = CatBoostClassifier(
        loss_function="Logloss", eval_metric="Logloss", random_seed=RANDOM_STATE,
        iterations=3000, early_stopping_rounds=50, use_best_model=True, verbose=100,
        **es_params, **gpu_params,
    )
    es_model.fit(X_fit, y_fit, eval_set=(X_es_val, y_es_val))
    final_iterations = es_model.get_best_iteration() + 1
    print(f"Early-stopping found optimal iterations = {final_iterations} "
          f"(ceiling was 3000, patience 50 rounds)")
    print(f"Stage-3 wall time: {(time.time() - t0) / 60:.1f} min")

    final_params = dict(best_params)
    final_params["iterations"] = final_iterations
    final_model = CatBoostClassifier(
        loss_function="Logloss", eval_metric="Logloss", random_seed=RANDOM_STATE,
        verbose=False, **final_params, **gpu_params,
    )

    print(f"\nFinal tuned hyperparameters: {final_params}")
    return final_model, final_params


# ---------------------------------------------------------------------
# 4. EVALUATION -- 5-fold CV with tqdm, GPU-or-fallback (5 large fits)
# ---------------------------------------------------------------------
def evaluate(model_params, gpu_params, X, y, cv_splits=5):
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
    accs, precs, recs, f1s, aucs = [], [], [], [], []
    t0 = time.time()

    for train_idx, test_idx in tqdm(list(cv.split(X, y)), desc="Stage 4: Final 5-fold CV"):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        model = CatBoostClassifier(
            loss_function="Logloss", eval_metric="Logloss", random_seed=RANDOM_STATE,
            verbose=False, **model_params, **gpu_params,
        )
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        y_proba = model.predict_proba(X_te)[:, 1]

        accs.append(accuracy_score(y_te, y_pred))
        precs.append(precision_score(y_te, y_pred))
        recs.append(recall_score(y_te, y_pred))
        f1s.append(f1_score(y_te, y_pred))
        aucs.append(roc_auc_score(y_te, y_proba))

    print(f"Stage-4 wall time: {(time.time() - t0) / 60:.1f} min")
    return {
        "Accuracy": (np.mean(accs), np.std(accs)),
        "Precision": (np.mean(precs), np.std(precs)),
        "Recall": (np.mean(recs), np.std(recs)),
        "F1-Score": (np.mean(f1s), np.std(f1s)),
        "AUC": (np.mean(aucs), np.std(aucs)),
    }


# ---------------------------------------------------------------------
# 5. COMPARISON TABLE
# ---------------------------------------------------------------------
REFERENCE_RESULTS = {
    "Alkhater (2026) - reported XGBoost": {
        "Accuracy": 0.970, "Precision": 0.960, "Recall": 0.960,
        "F1-Score": 0.960, "AUC": 0.980,
    },
    "Kasongo & Sun (2020) - XGBoost (feature-selection only)": {
        "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan,
        "F1-Score": np.nan, "AUC": np.nan,
    },
    "Mohale & Obagbuwa (2025) - reported XGBoost": {
        "Accuracy": 0.8687, "Precision": 0.850, "Recall": 0.880,
        "F1-Score": 0.860, "AUC": 0.930,
    },
}


def build_comparison_table(my_results, out_csv=os.path.join(OUT_DIR, "comparison_table_catboost.csv")):
    rows = []
    for name, metrics in REFERENCE_RESULTS.items():
        row = {"Model": name}
        row.update(metrics)
        rows.append(row)

    my_row = {"Model": "Proposed Tuned CatBoost (ALL features, no selection)"}
    for k, (mean, _std) in my_results.items():
        my_row[k] = round(mean, 4)
    rows.append(my_row)

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(df.to_string(index=False))
    return df


# ---------------------------------------------------------------------
# 6. GRAPHS
# ---------------------------------------------------------------------
def plot_comparison_bar(df, out_path=os.path.join(OUT_DIR, "comparison_bar_catboost.png")):
    metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]
    x = np.arange(len(df))
    width = 0.2
    fig, ax = plt.subplots(figsize=(11, 6))
    for i, m in enumerate(metrics):
        ax.bar(x + i * width, df[m], width, label=m)
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(df["Model"], rotation=20, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title("CatBoost -- Proposed Model vs. Reference Papers")
    ax.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_confusion_matrix(model, X_test, y_test, out_path=os.path.join(OUT_DIR, "confusion_matrix_catboost.png")):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Malicious"])
    fig, ax = plt.subplots(figsize=(5, 5))
    disp.plot(ax=ax, cmap="Purples", values_format="d")
    ax.set_title("Confusion Matrix -- Tuned CatBoost")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_roc(model, X_test, y_test, out_path=os.path.join(OUT_DIR, "roc_curve_catboost.png")):
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, color="purple", label=f"Tuned CatBoost (AUC = {auc:.3f})")
    plt.plot([0, 1], [0, 1], "--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve -- Tuned CatBoost (All Features)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_feature_importance(model, feature_names, out_path=os.path.join(OUT_DIR, "feature_importance_catboost.png")):
    importances = model.get_feature_importance()
    order = np.argsort(importances)[::-1]
    plt.figure(figsize=(9, max(6, 0.28 * len(order))))
    plt.barh([feature_names[i] for i in order][::-1], importances[order][::-1], color="darkviolet")
    plt.xlabel("PredictionValuesChange Importance")
    plt.title(f"CatBoost Feature Importance -- ALL {len(feature_names)} Features Used")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


# ---------------------------------------------------------------------
# 7. MAIN
# ---------------------------------------------------------------------
if __name__ == "__main__":
    TRAIN_CSV = "UNSW_NB15_training-set.csv"
    TEST_CSV = "UNSW_NB15_testing-set.csv"

    pipeline_start = time.time()

    gpu_params = detect_and_configure_gpu()

    print("\nLoading data...")
    df_train, df_test = load_unsw_nb15(train_path=TRAIN_CSV, test_path=TEST_CSV)

    print("Preprocessing (no feature selection -- all columns kept, "
          "encoders/scaler fit on train only)...")
    X_train, X_test, y_train, y_test, scaler, encoders = preprocess_train_test(df_train, df_test)

    X_full = pd.concat([X_train, X_test], axis=0).reset_index(drop=True)
    y_full = pd.concat([y_train, y_test], axis=0).reset_index(drop=True)

    print("\nHyperparameter tuning (Random -> Grid, both CPU; "
          "then GPU-or-fallback early-stopping refinement)...")
    t0 = time.time()
    best_model, best_params = tune_catboost(X_train, y_train, gpu_params)
    print(f"\nTotal tuning time: {(time.time() - t0) / 60:.1f} min")

    print("\n5-fold CV evaluation of the tuned model on the FULL dataset...")
    t0 = time.time()
    cv_results = evaluate(best_params, gpu_params, X_full, y_full, cv_splits=5)
    print(f"5-fold evaluation took {(time.time() - t0) / 60:.1f} min")
    for k, (m, s) in cv_results.items():
        print(f"{k}: {m:.4f} +/- {s:.4f}")

    print("\nBuilding comparison table against the 3 reference papers...")
    comp_df = build_comparison_table(cv_results)

    print("\nGenerating graphs...")
    best_model.fit(X_train, y_train)  # final fit for the plots below
    plot_comparison_bar(comp_df)
    plot_confusion_matrix(best_model, X_test, y_test)
    plot_roc(best_model, X_test, y_test)
    plot_feature_importance(best_model, X_full.columns.tolist())

    print(f"\nAll outputs saved to ./{OUT_DIR}/ (with *_catboost suffix)")
    total_min = (time.time() - pipeline_start) / 60
    print(f"\nTOTAL PIPELINE TIME: {total_min:.1f} min "
          f"({'within' if total_min <= 120 else 'OVER'} the 1-2 hour target)")

# logistic regression new implementation

In [1]:
"""
CatBoost (Binary Classification) on UNSW-NB15 -- FULL FEATURE SET, FAST VERSION
========================================================================
Same preprocessing pipeline as the XGBoost/Logistic Regression companion
scripts (NO feature selection -- all original columns kept; encoders/
scaler fit on train only, applied to test).

WHY THIS VERSION DOESN'T TAKE HOURS ANYMORE
---------------------------------------------
CatBoost's GPU mode has a well-documented FIXED overhead of roughly 60
seconds per .fit() call, independent of data size (CatBoost GitHub
issues #2084, #1034, #629 -- the third one reports outright hangs after
many sequential fits). A hyperparameter search does dozens/hundreds of
small fits, so running that search on GPU is actively counterproductive
-- you pay ~60s of setup for every single fit, no matter how small.

The fix:
  1. Hyperparameter search (Stages 1-2) runs on CPU via a manual search
     loop (not sklearn's *SearchCV), using all CPU cores
     (thread_count=-1). This is what was silently taking hours.
  2. GPU (if detected) is used ONLY for Stage 3 (early-stopping
     refinement on the full training set) and Stage 4 (final 5-fold CV)
     -- a handful of large, long-running fits, which is what GPU
     actually accelerates well.
  3. tqdm progress bars on every loop (Stage 1, Stage 2, final CV) so
     you see live progress instead of a silent gap.
  4. A hard wall-clock TIME BUDGET on Stages 1 and 2 (25 min each by
     default) -- if the search isn't done by then, it stops and keeps
     the best candidate found so far. This guarantees the pipeline
     cannot silently run for hours, by construction, not just by hope.

Expected total runtime: roughly 35-75 minutes on a typical Kaggle GPU
session (T4), safely inside a 1-2 hour ceiling -- printed timings after
every stage let you see exactly where the time goes.

Reference results (the numbers to beat -- Kasongo & Sun report no
standalone XGBoost number for UNSW-NB15, so they're marked N/A):

  [1] Alkhater, N. (2026). Computers, 15(5), 285.
      -> XGBoost (default config, 5-fold CV, 48 features):
         Acc=0.97+-0.005, Prec=0.96+-0.006, Recall=0.96+-0.005,
         F1=0.96+-0.006, AUC=0.980+-0.003

  [2] Kasongo, S.M. & Sun, Y. (2020). J Big Data, 7:105.
      -> XGBoost used only for feature-importance/selection -- N/A.

  [3] Mohale, V.Z. & Obagbuwa, I.C. (2025). Front. Comput. Sci., 7:1520741.
      -> XGBoost: Acc=86.87%, Precision=0.85, Recall=0.88, F1=0.86,
         ROC-AUC=0.93, FPR=0.08, FNR=0.11

Requires: pip install catboost tqdm   (use --break-system-packages if needed)
"""

import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
)

from catboost import CatBoostClassifier, CatBoostError
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

SEARCH_SAMPLE_SIZE = 50_000       # rows used ONLY for hyperparameter search
STAGE_TIME_BUDGET_S = 25 * 60     # hard cap per search stage: 25 minutes


# ---------------------------------------------------------------------
# 0. GPU DETECTION (used only for Stages 3-4, the large fits)
# ---------------------------------------------------------------------
def detect_and_configure_gpu():
    dummy_X = np.random.rand(200, 5)
    dummy_y = np.random.randint(0, 2, 200)
    try:
        probe = CatBoostClassifier(task_type="GPU", devices="0", iterations=5, verbose=False)
        probe.fit(dummy_X, dummy_y)
        print("=" * 60)
        print("GPU DETECTED -- Stages 3-4 (the large fits) will use it.")
        print("Stages 1-2 (hyperparameter search) use CPU regardless --")
        print("GPU has ~60s fixed overhead per fit, which makes it SLOWER")
        print("for many small search fits (see catboost issue #2084).")
        print("=" * 60)
        return {"task_type": "GPU", "devices": "0"}
    except CatBoostError as e:
        print("=" * 60)
        print("GPU NOT DETECTED -- Stages 3-4 will also run on CPU.")
        print(f"(CatBoost reported: {str(e)[:120]})")
        print("If you're on Kaggle: Settings -> Accelerator -> GPU T4 x2")
        print("=" * 60)
        return {"task_type": "CPU", "thread_count": -1}


# ---------------------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------------------
def load_unsw_nb15(train_path=None, test_path=None):
    df_train = pd.read_csv("/kaggle/input/datasets/ajeetkumar20/unsw-nb15-v1-1/UNSW_NB15_training-set.csv")
    df_test = pd.read_csv("/kaggle/input/datasets/ajeetkumar20/unsw-nb15-v1-1/UNSW_NB15_testing-set.csv")
    return df_train, df_test


# ---------------------------------------------------------------------
# 2. PREPROCESSING (unchanged -- no feature selection, encoders/scaler
#    fit on train only)
# ---------------------------------------------------------------------
def preprocess_train_test(df_train, df_test):
    train = df_train.copy()
    test = df_test.copy()

    for id_col in ["id", "ID", "Id"]:
        if id_col in train.columns:
            train.drop(columns=[id_col], inplace=True)
        if id_col in test.columns:
            test.drop(columns=[id_col], inplace=True)

    y_train = train["label"].astype(int)
    y_test = test["label"].astype(int)
    train.drop(columns=["label"], inplace=True)
    test.drop(columns=["label"], inplace=True)

    if "attack_cat" in train.columns:
        train.drop(columns=["attack_cat"], inplace=True)
    if "attack_cat" in test.columns:
        test.drop(columns=["attack_cat"], inplace=True)

    encoders = {}
    categorical_cols = train.select_dtypes(include=["object"]).columns
    for col in categorical_cols:
        le = LabelEncoder()
        combined = pd.concat([train[col].astype(str), test[col].astype(str)])
        le.fit(combined)
        train[col] = le.transform(train[col].astype(str))
        test[col] = le.transform(test[col].astype(str))
        encoders[col] = le

    scaler = MinMaxScaler()
    X_train = pd.DataFrame(scaler.fit_transform(train), columns=train.columns)
    X_test = pd.DataFrame(scaler.transform(test), columns=test.columns)

    print(f"Training features : {X_train.shape}")
    print(f"Testing features  : {X_test.shape}")

    return X_train, X_test, y_train, y_test, scaler, encoders


# ---------------------------------------------------------------------
# 3. HYPERPARAMETER TUNING -- manual search loop, CPU-only, tqdm +
#    hard time budget, then GPU-or-fallback for the two large fits.
# ---------------------------------------------------------------------
def make_search_subsample(X_train, y_train, sample_size=SEARCH_SAMPLE_SIZE, random_state=RANDOM_STATE):
    if len(X_train) <= sample_size:
        return X_train, y_train
    X_search, _, y_search, _ = train_test_split(
        X_train, y_train, train_size=sample_size, stratify=y_train, random_state=random_state
    )
    print(f"Hyperparameter search will use a stratified subsample: "
          f"{len(X_search)} / {len(X_train)} rows ({100 * len(X_search) / len(X_train):.1f}%)")
    return X_search, y_search


def run_candidate_search(X, y, candidates, cv_folds, base_params, time_budget_s, desc):
    """Evaluate a list of hyperparameter-dict candidates via k-fold CV,
    with a live tqdm progress bar and a hard wall-clock time budget --
    if the budget is hit mid-search, keep whatever's been found so far
    instead of running unbounded."""
    skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=RANDOM_STATE)
    results = []
    t_start = time.time()
    total_fits = len(candidates) * cv_folds

    pbar = tqdm(total=total_fits, desc=desc)
    stopped_early = False
    for cand in candidates:
        if time.time() - t_start > time_budget_s:
            stopped_early = True
            break
        fold_scores = []
        for train_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            model = CatBoostClassifier(**base_params, **cand, verbose=False)
            model.fit(X_tr, y_tr)
            preds = model.predict(X_val)
            fold_scores.append(f1_score(y_val, preds))
            pbar.update(1)
        results.append({"params": cand, "mean_f1": float(np.mean(fold_scores))})
    pbar.close()

    if stopped_early:
        print(f"[{desc}] Time budget ({time_budget_s / 60:.0f} min) reached -- "
              f"stopped after {len(results)}/{len(candidates)} candidates. "
              f"Using the best one found so far.")

    results.sort(key=lambda r: r["mean_f1"], reverse=True)
    print(f"[{desc}] Best: {results[0]['params']}  (CV F1 = {results[0]['mean_f1']:.4f})")
    return results[0]["params"], results[0]["mean_f1"]


def widen(v, lo_bound, step, as_int=True):
    vals = {v - step, v, v + step}
    vals = {max(lo_bound, x) for x in vals}
    return sorted({int(x) if as_int else round(x, 4) for x in vals})


def tune_catboost(X_train, y_train, gpu_params):
    X_search, y_search = make_search_subsample(X_train, y_train)

    cpu_search_params = {
        "task_type": "CPU", "thread_count": -1,
        "loss_function": "Logloss", "eval_metric": "Logloss",
        "random_seed": RANDOM_STATE,
    }

    # ---- Stage 1: random search over the full space, CPU, 3-fold ----
    param_space = {
        "iterations": [150, 250, 400],
        "depth": [4, 5, 6, 7, 8, 9, 10],
        "learning_rate": [0.02, 0.03, 0.05, 0.08, 0.1, 0.15, 0.2],
        "l2_leaf_reg": [1, 3, 5, 7, 9, 12, 15],
        "border_count": [32, 64, 128, 254],
        "bagging_temperature": [0, 0.2, 0.5, 1.0, 2.0],
        "random_strength": [0, 0.5, 1, 2, 5],
    }
    rng = np.random.RandomState(RANDOM_STATE)
    n_random_candidates = 25
    candidates = [
        {k: rng.choice(v).item() if hasattr(rng.choice(v), "item") else rng.choice(v)
         for k, v in param_space.items()}
        for _ in range(n_random_candidates)
    ]

    t0 = time.time()
    best_rand, _ = run_candidate_search(
        X_search, y_search, candidates, cv_folds=3, base_params=cpu_search_params,
        time_budget_s=STAGE_TIME_BUDGET_S, desc="Stage 1: Random search (CPU)"
    )
    print(f"Stage-1 wall time: {(time.time() - t0) / 60:.1f} min")

    # ---- Stage 2: narrow grid around the Stage-1 winner, CPU, 3-fold ----
    grid_candidates = []
    for it in widen(best_rand["iterations"], 100, 100):
        for d in widen(best_rand["depth"], 2, 1):
            for lr in widen(best_rand["learning_rate"], 0.02, 0.02, as_int=False):
                grid_candidates.append({
                    "iterations": it, "depth": d, "learning_rate": lr,
                    "l2_leaf_reg": best_rand["l2_leaf_reg"],
                    "border_count": best_rand["border_count"],
                    "bagging_temperature": best_rand["bagging_temperature"],
                    "random_strength": best_rand["random_strength"],
                })

    t0 = time.time()
    best_params, _ = run_candidate_search(
        X_search, y_search, grid_candidates, cv_folds=3, base_params=cpu_search_params,
        time_budget_s=STAGE_TIME_BUDGET_S, desc="Stage 2: Grid refine (CPU)"
    )
    print(f"Stage-2 wall time: {(time.time() - t0) / 60:.1f} min")

    # ---- Stage 3: early-stopping refinement of "iterations" on the FULL
    #      training set, using GPU if available (this is the first fit
    #      large/long enough for GPU to actually help). ----
    t0 = time.time()
    X_fit, X_es_val, y_fit, y_es_val = train_test_split(
        X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE
    )
    es_params = {k: v for k, v in best_params.items() if k != "iterations"}
    es_model = CatBoostClassifier(
        loss_function="Logloss", eval_metric="Logloss", random_seed=RANDOM_STATE,
        iterations=3000, early_stopping_rounds=50, use_best_model=True, verbose=100,
        **es_params, **gpu_params,
    )
    es_model.fit(X_fit, y_fit, eval_set=(X_es_val, y_es_val))
    final_iterations = es_model.get_best_iteration() + 1
    print(f"Early-stopping found optimal iterations = {final_iterations} "
          f"(ceiling was 3000, patience 50 rounds)")
    print(f"Stage-3 wall time: {(time.time() - t0) / 60:.1f} min")

    final_params = dict(best_params)
    final_params["iterations"] = final_iterations
    final_model = CatBoostClassifier(
        loss_function="Logloss", eval_metric="Logloss", random_seed=RANDOM_STATE,
        verbose=False, **final_params, **gpu_params,
    )

    print(f"\nFinal tuned hyperparameters: {final_params}")
    return final_model, final_params


# ---------------------------------------------------------------------
# 4. EVALUATION -- 5-fold CV with tqdm, GPU-or-fallback (5 large fits)
# ---------------------------------------------------------------------
def evaluate(model_params, gpu_params, X, y, cv_splits=5):
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
    accs, precs, recs, f1s, aucs = [], [], [], [], []
    t0 = time.time()

    for train_idx, test_idx in tqdm(list(cv.split(X, y)), desc="Stage 4: Final 5-fold CV"):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        model = CatBoostClassifier(
            loss_function="Logloss", eval_metric="Logloss", random_seed=RANDOM_STATE,
            verbose=False, **model_params, **gpu_params,
        )
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        y_proba = model.predict_proba(X_te)[:, 1]

        accs.append(accuracy_score(y_te, y_pred))
        precs.append(precision_score(y_te, y_pred))
        recs.append(recall_score(y_te, y_pred))
        f1s.append(f1_score(y_te, y_pred))
        aucs.append(roc_auc_score(y_te, y_proba))

    print(f"Stage-4 wall time: {(time.time() - t0) / 60:.1f} min")
    return {
        "Accuracy": (np.mean(accs), np.std(accs)),
        "Precision": (np.mean(precs), np.std(precs)),
        "Recall": (np.mean(recs), np.std(recs)),
        "F1-Score": (np.mean(f1s), np.std(f1s)),
        "AUC": (np.mean(aucs), np.std(aucs)),
    }


# ---------------------------------------------------------------------
# 5. COMPARISON TABLE
# ---------------------------------------------------------------------
REFERENCE_RESULTS = {
    "Alkhater (2026) - reported XGBoost": {
        "Accuracy": 0.970, "Precision": 0.960, "Recall": 0.960,
        "F1-Score": 0.960, "AUC": 0.980,
    },
    "Kasongo & Sun (2020) - XGBoost (feature-selection only)": {
        "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan,
        "F1-Score": np.nan, "AUC": np.nan,
    },
    "Mohale & Obagbuwa (2025) - reported XGBoost": {
        "Accuracy": 0.8687, "Precision": 0.850, "Recall": 0.880,
        "F1-Score": 0.860, "AUC": 0.930,
    },
}


def build_comparison_table(my_results, out_csv=os.path.join(OUT_DIR, "comparison_table_catboost.csv")):
    rows = []
    for name, metrics in REFERENCE_RESULTS.items():
        row = {"Model": name}
        row.update(metrics)
        rows.append(row)

    my_row = {"Model": "Proposed Tuned CatBoost (ALL features, no selection)"}
    for k, (mean, _std) in my_results.items():
        my_row[k] = round(mean, 4)
    rows.append(my_row)

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(df.to_string(index=False))
    return df


# ---------------------------------------------------------------------
# 6. GRAPHS
# ---------------------------------------------------------------------
def plot_comparison_bar(df, out_path=os.path.join(OUT_DIR, "comparison_bar_catboost.png")):
    metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]
    x = np.arange(len(df))
    width = 0.2
    fig, ax = plt.subplots(figsize=(11, 6))
    for i, m in enumerate(metrics):
        ax.bar(x + i * width, df[m], width, label=m)
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(df["Model"], rotation=20, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title("CatBoost -- Proposed Model vs. Reference Papers")
    ax.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_confusion_matrix(model, X_test, y_test, out_path=os.path.join(OUT_DIR, "confusion_matrix_catboost.png")):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Malicious"])
    fig, ax = plt.subplots(figsize=(5, 5))
    disp.plot(ax=ax, cmap="Purples", values_format="d")
    ax.set_title("Confusion Matrix -- Tuned CatBoost")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_roc(model, X_test, y_test, out_path=os.path.join(OUT_DIR, "roc_curve_catboost.png")):
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, color="purple", label=f"Tuned CatBoost (AUC = {auc:.3f})")
    plt.plot([0, 1], [0, 1], "--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve -- Tuned CatBoost (All Features)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_feature_importance(model, feature_names, out_path=os.path.join(OUT_DIR, "feature_importance_catboost.png")):
    importances = model.get_feature_importance()
    order = np.argsort(importances)[::-1]
    plt.figure(figsize=(9, max(6, 0.28 * len(order))))
    plt.barh([feature_names[i] for i in order][::-1], importances[order][::-1], color="darkviolet")
    plt.xlabel("PredictionValuesChange Importance")
    plt.title(f"CatBoost Feature Importance -- ALL {len(feature_names)} Features Used")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


# ---------------------------------------------------------------------
# 7. MAIN
# ---------------------------------------------------------------------
if __name__ == "__main__":
    TRAIN_CSV = "UNSW_NB15_training-set.csv"
    TEST_CSV = "UNSW_NB15_testing-set.csv"

    pipeline_start = time.time()

    gpu_params = detect_and_configure_gpu()

    print("\nLoading data...")
    df_train, df_test = load_unsw_nb15(train_path=TRAIN_CSV, test_path=TEST_CSV)

    print("Preprocessing (no feature selection -- all columns kept, "
          "encoders/scaler fit on train only)...")
    X_train, X_test, y_train, y_test, scaler, encoders = preprocess_train_test(df_train, df_test)

    X_full = pd.concat([X_train, X_test], axis=0).reset_index(drop=True)
    y_full = pd.concat([y_train, y_test], axis=0).reset_index(drop=True)

    print("\nHyperparameter tuning (Random -> Grid, both CPU; "
          "then GPU-or-fallback early-stopping refinement)...")
    t0 = time.time()
    best_model, best_params = tune_catboost(X_train, y_train, gpu_params)
    print(f"\nTotal tuning time: {(time.time() - t0) / 60:.1f} min")

    print("\n5-fold CV evaluation of the tuned model on the FULL dataset...")
    t0 = time.time()
    cv_results = evaluate(best_params, gpu_params, X_full, y_full, cv_splits=5)
    print(f"5-fold evaluation took {(time.time() - t0) / 60:.1f} min")
    for k, (m, s) in cv_results.items():
        print(f"{k}: {m:.4f} +/- {s:.4f}")

    print("\nBuilding comparison table against the 3 reference papers...")
    comp_df = build_comparison_table(cv_results)

    print("\nGenerating graphs...")
    best_model.fit(X_train, y_train)  # final fit for the plots below
    plot_comparison_bar(comp_df)
    plot_confusion_matrix(best_model, X_test, y_test)
    plot_roc(best_model, X_test, y_test)
    plot_feature_importance(best_model, X_full.columns.tolist())

    print(f"\nAll outputs saved to ./{OUT_DIR}/ (with *_catboost suffix)")
    total_min = (time.time() - pipeline_start) / 60
    print(f"\nTOTAL PIPELINE TIME: {total_min:.1f} min "
          f"({'within' if total_min <= 120 else 'OVER'} the 1-2 hour target)")

GPU DETECTED -- Stages 3-4 (the large fits) will use it.
Stages 1-2 (hyperparameter search) use CPU regardless --
GPU has ~60s fixed overhead per fit, which makes it SLOWER
for many small search fits (see catboost issue #2084).

Loading data...
Preprocessing (no feature selection -- all columns kept, encoders/scaler fit on train only)...
Training features : (175341, 42)
Testing features  : (82332, 42)

Hyperparameter tuning (Random -> Grid, both CPU; then GPU-or-fallback early-stopping refinement)...
Hyperparameter search will use a stratified subsample: 50000 / 175341 rows (28.5%)


Stage 1: Random search (CPU):   0%|          | 0/75 [00:00<?, ?it/s]

[Stage 1: Random search (CPU)] Best: {'iterations': 400, 'depth': 7, 'learning_rate': 0.08, 'l2_leaf_reg': 1, 'border_count': 254, 'bagging_temperature': 0.0, 'random_strength': 2.0}  (CV F1 = 0.9652)
Stage-1 wall time: 4.6 min


Stage 2: Grid refine (CPU):   0%|          | 0/81 [00:00<?, ?it/s]

[Stage 2: Grid refine (CPU)] Best: {'iterations': 400, 'depth': 8, 'learning_rate': 0.06, 'l2_leaf_reg': 1, 'border_count': 254, 'bagging_temperature': 0.0, 'random_strength': 2.0}  (CV F1 = 0.9654)
Stage-2 wall time: 7.7 min
0:	learn: 0.5297201	test: 0.5291954	best: 0.5291954 (0)	total: 96.8ms	remaining: 4m 50s
100:	learn: 0.1001061	test: 0.1010587	best: 0.1010587 (100)	total: 1.09s	remaining: 31.4s
200:	learn: 0.0917952	test: 0.0952620	best: 0.0952620 (200)	total: 1.95s	remaining: 27.1s
300:	learn: 0.0840997	test: 0.0911480	best: 0.0911480 (300)	total: 2.82s	remaining: 25.3s
400:	learn: 0.0780258	test: 0.0887022	best: 0.0887022 (400)	total: 3.68s	remaining: 23.9s
500:	learn: 0.0728860	test: 0.0873079	best: 0.0873079 (500)	total: 4.55s	remaining: 22.7s
600:	learn: 0.0683667	test: 0.0860232	best: 0.0860060 (597)	total: 5.43s	remaining: 21.7s
700:	learn: 0.0644196	test: 0.0852576	best: 0.0852373 (692)	total: 6.3s	remaining: 20.7s
800:	learn: 0.0608957	test: 0.0842549	best: 0.0842427 (79

Stage 4: Final 5-fold CV:   0%|          | 0/5 [00:00<?, ?it/s]

Stage-4 wall time: 1.3 min
5-fold evaluation took 1.3 min
Accuracy: 0.9526 +/- 0.0007
Precision: 0.9664 +/- 0.0011
Recall: 0.9591 +/- 0.0007
F1-Score: 0.9627 +/- 0.0006
AUC: 0.9929 +/- 0.0002

Building comparison table against the 3 reference papers...
                                                  Model  Accuracy  Precision  Recall  F1-Score    AUC
                     Alkhater (2026) - reported XGBoost    0.9700     0.9600  0.9600    0.9600 0.9800
Kasongo & Sun (2020) - XGBoost (feature-selection only)       NaN        NaN     NaN       NaN    NaN
            Mohale & Obagbuwa (2025) - reported XGBoost    0.8687     0.8500  0.8800    0.8600 0.9300
   Proposed Tuned CatBoost (ALL features, no selection)    0.9526     0.9664  0.9591    0.9627 0.9929

Generating graphs...

All outputs saved to ./outputs/ (with *_catboost suffix)

TOTAL PIPELINE TIME: 14.1 min (within the 1-2 hour target)


# linear Regression

In [1]:
"""
Logistic Regression (Binary Classification) on UNSW-NB15 -- FULL FEATURE SET, FAST VERSION
========================================================================
Same preprocessing pipeline as the XGBoost/CatBoost companion scripts
(NO feature selection -- all original columns kept; encoders/scaler fit
on train only, applied to test).

WHY THIS VERSION DOESN'T TAKE HOURS ANYMORE
---------------------------------------------
Measured directly (not assumed): with solver="saga" on MinMax-scaled
features, L1-penalized fits at moderate-to-high C need 900-1600+
iterations to converge -- 95-170 SECONDS for a single fit. The original
search runs 200+ such fits across Stage 1 alone. That's the entire
"several hours."

The fix uses the solver actually suited to each penalty type:
  - L1      -> liblinear   (13x faster than saga at C=100 in testing)
  - L2      -> lbfgs       (240x faster than saga at C=100 in testing)
  - elastic -> saga        (the only solver that supports true L1/L2
                             mixing -- but it wasn't the slow case here)
Search-time fits also use a capped max_iter=1000 (just for RANKING
candidates relatively); the one final production model gets a full
max_iter=5000 for complete convergence -- so no quality is traded away,
only wasted iterations on throwaway search candidates.

Also added: a manual search loop (not sklearn's *SearchCV) so we get
live tqdm progress bars and a hard wall-clock time budget per stage --
if a stage isn't done in 20 minutes, it stops and keeps the best found
so far, guaranteeing the pipeline can't silently run for hours no
matter what.

Reference results (the numbers to beat):

  [1] Hakke, D.G. et al. (2025). Int. J. Applied Mathematics, 38(3s), 447.
      -> Logistic Regression: Accuracy=88.69%, Precision(Attack)=0.85,
         Recall(Attack)=0.99, F1(Attack)=0.91

  [2] Kasongo, S.M. & Sun, Y. (2020). J Big Data, 7:105.
      -> Logistic Regression (42 features, no selection):
         Test Acc=79.59%, Precision=73.32%, Recall=98.94%, F1=84.22%

Requires: pip install tqdm   (use --break-system-packages if needed)
"""

import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
)

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

STAGE_TIME_BUDGET_S = 20 * 60   # hard cap per search stage: 20 minutes
SEARCH_MAX_ITER = 1000          # capped iterations DURING search only
FINAL_MAX_ITER = 5000           # full iterations for the one production fit

# The actual fix: the right solver per penalty type, not one-size-fits-all "saga"
SOLVER_FOR_PENALTY = {"l1": "liblinear", "l2": "lbfgs", "elasticnet": "saga"}


# ---------------------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------------------
def load_unsw_nb15(train_path=None, test_path=None):
    df_train = pd.read_csv("/kaggle/input/datasets/ajeetkumar20/unsw-nb15-v1-1/UNSW_NB15_training-set.csv")
    df_test = pd.read_csv("/kaggle/input/datasets/ajeetkumar20/unsw-nb15-v1-1/UNSW_NB15_testing-set.csv")
    return df_train, df_test


# ---------------------------------------------------------------------
# 2. PREPROCESSING (unchanged)
# ---------------------------------------------------------------------
def preprocess_train_test(df_train, df_test):
    train = df_train.copy()
    test = df_test.copy()

    for id_col in ["id", "ID", "Id"]:
        if id_col in train.columns:
            train.drop(columns=[id_col], inplace=True)
        if id_col in test.columns:
            test.drop(columns=[id_col], inplace=True)

    y_train = train["label"].astype(int)
    y_test = test["label"].astype(int)
    train.drop(columns=["label"], inplace=True)
    test.drop(columns=["label"], inplace=True)

    if "attack_cat" in train.columns:
        train.drop(columns=["attack_cat"], inplace=True)
    if "attack_cat" in test.columns:
        test.drop(columns=["attack_cat"], inplace=True)

    encoders = {}
    categorical_cols = train.select_dtypes(include=["object"]).columns
    for col in categorical_cols:
        le = LabelEncoder()
        combined = pd.concat([train[col].astype(str), test[col].astype(str)])
        le.fit(combined)
        train[col] = le.transform(train[col].astype(str))
        test[col] = le.transform(test[col].astype(str))
        encoders[col] = le

    scaler = MinMaxScaler()
    X_train = pd.DataFrame(scaler.fit_transform(train), columns=train.columns)
    X_test = pd.DataFrame(scaler.transform(test), columns=test.columns)

    print(f"Training features : {X_train.shape}")
    print(f"Testing features  : {X_test.shape}")

    return X_train, X_test, y_train, y_test, scaler, encoders


# ---------------------------------------------------------------------
# 3. THRESHOLD OPTIMIZATION HELPER (unchanged)
# ---------------------------------------------------------------------
def optimize_threshold(y_true, y_proba, metric="f1"):
    best_thr, best_score = 0.5, -1.0
    for thr in np.arange(0.01, 1.00, 0.01):
        y_pred = (y_proba >= thr).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0) if metric == "f1" \
            else accuracy_score(y_true, y_pred)
        if score > best_score:
            best_score, best_thr = score, thr
    return best_thr, best_score


# ---------------------------------------------------------------------
# 4. HYPERPARAMETER TUNING -- manual search loop, penalty-appropriate
#    solver, capped search max_iter, tqdm + hard time budget.
# ---------------------------------------------------------------------
def run_candidate_search(X, y, candidates, cv_folds, time_budget_s, desc):
    """Evaluate a list of hyperparameter-dict candidates via k-fold CV,
    with a live tqdm progress bar and a hard wall-clock time budget."""
    skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=RANDOM_STATE)
    results = []
    t_start = time.time()
    total_fits = len(candidates) * cv_folds

    pbar = tqdm(total=total_fits, desc=desc)
    stopped_early = False
    for cand in candidates:
        if time.time() - t_start > time_budget_s:
            stopped_early = True
            break
        solver = SOLVER_FOR_PENALTY[cand["penalty"]]
        fit_kwargs = {k: v for k, v in cand.items()}
        fold_scores = []
        for train_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            model = LogisticRegression(
                solver=solver, max_iter=SEARCH_MAX_ITER, tol=1e-4,
                random_state=RANDOM_STATE, **fit_kwargs
            )
            model.fit(X_tr, y_tr)
            preds = model.predict(X_val)
            fold_scores.append(f1_score(y_val, preds, zero_division=0))
            pbar.update(1)
        results.append({"params": cand, "solver": solver, "mean_f1": float(np.mean(fold_scores))})
    pbar.close()

    if stopped_early:
        print(f"[{desc}] Time budget ({time_budget_s / 60:.0f} min) reached -- "
              f"stopped after {len(results)}/{len(candidates)} candidates. "
              f"Using the best one found so far.")

    results.sort(key=lambda r: r["mean_f1"], reverse=True)
    best = results[0]
    print(f"[{desc}] Best: {best['params']} (solver={best['solver']})  CV F1 = {best['mean_f1']:.4f}")
    return best["params"]


def widen_log(v, factor=3):
    return sorted({round(v / factor, 5), v, round(v * factor, 5)})


def tune_logreg(X_train, y_train):
    rng = np.random.RandomState(RANDOM_STATE)

    # C capped at 10 -- the pathological slow zone was C=50/100, and
    # values that high rarely help generalization anyway (near-zero
    # regularization). Dropping them removes the worst-case fits
    # entirely rather than just making them individually faster.
    param_space = [
        {"penalty": "l1", "C": [0.001, 0.01, 0.1, 0.5, 1, 2, 5, 10], "class_weight": [None, "balanced"]},
        {"penalty": "l2", "C": [0.001, 0.01, 0.1, 0.5, 1, 2, 5, 10], "class_weight": [None, "balanced"]},
        {"penalty": "elasticnet", "C": [0.001, 0.01, 0.1, 0.5, 1, 2, 5, 10],
         "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9], "class_weight": [None, "balanced"]},
    ]

    n_candidates_total = 40
    candidates = []
    for _ in range(n_candidates_total):
        space = param_space[rng.randint(len(param_space))]
        cand = {"penalty": space["penalty"]}
        cand["C"] = float(rng.choice(space["C"]))
        cand["class_weight"] = rng.choice(space["class_weight"])
        if cand["class_weight"] is None:
            cand["class_weight"] = None  # rng.choice can wrap None oddly; keep explicit
        if space["penalty"] == "elasticnet":
            cand["l1_ratio"] = float(rng.choice(space["l1_ratio"]))
        candidates.append(cand)

    t0 = time.time()
    best_rand = run_candidate_search(
        X_train, y_train, candidates, cv_folds=5,
        time_budget_s=STAGE_TIME_BUDGET_S, desc="Stage 1: Random search"
    )
    print(f"Stage-1 wall time: {(time.time() - t0) / 60:.1f} min")

    # ---- Stage 2: narrow grid around the Stage-1 winner ----
    penalty = best_rand["penalty"]
    grid_candidates = []
    for c in widen_log(best_rand["C"]):
        cand = {"penalty": penalty, "C": c, "class_weight": best_rand["class_weight"]}
        if penalty == "elasticnet":
            v = best_rand["l1_ratio"]
            for lr in sorted({max(0.0, v - 0.15), v, min(1.0, v + 0.15)}):
                grid_candidates.append({**cand, "l1_ratio": lr})
        else:
            grid_candidates.append(cand)

    t0 = time.time()
    best_params = run_candidate_search(
        X_train, y_train, grid_candidates, cv_folds=5,
        time_budget_s=STAGE_TIME_BUDGET_S, desc="Stage 2: Grid refine"
    )
    print(f"Stage-2 wall time: {(time.time() - t0) / 60:.1f} min")

    # ---- Final model: full max_iter, no cap -- this is the only fit
    #      that needs to fully converge, and it's a single fit. ----
    final_solver = SOLVER_FOR_PENALTY[best_params["penalty"]]
    final_model = LogisticRegression(
        solver=final_solver, max_iter=FINAL_MAX_ITER, tol=1e-4,
        random_state=RANDOM_STATE, **best_params,
    )
    print(f"\nFinal tuned hyperparameters: {best_params}  (solver={final_solver}, max_iter={FINAL_MAX_ITER})")
    return final_model, best_params


# ---------------------------------------------------------------------
# 5. EVALUATION (unchanged design -- these are single fits, not search
#    fits, so they were never the bottleneck; now with tqdm on the CV loop)
# ---------------------------------------------------------------------
def _score(y_true, y_pred, y_proba):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0),
        "AUC": roc_auc_score(y_true, y_proba),
    }


def evaluate_cv(model, X, y, cv_splits=5):
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
    default_runs, tuned_runs, thresholds = [], [], []
    t0 = time.time()

    for train_idx, test_idx in tqdm(list(cv.split(X, y)), desc="Stage 3: Final 5-fold CV"):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        X_fit, X_thr, y_fit, y_thr = train_test_split(
            X_tr, y_tr, test_size=0.15, stratify=y_tr, random_state=RANDOM_STATE
        )

        model.fit(X_fit, y_fit)
        y_proba_thr = model.predict_proba(X_thr)[:, 1]
        best_thr, _ = optimize_threshold(y_thr, y_proba_thr)
        thresholds.append(best_thr)

        y_proba_te = model.predict_proba(X_te)[:, 1]
        default_runs.append(_score(y_te, (y_proba_te >= 0.5).astype(int), y_proba_te))
        tuned_runs.append(_score(y_te, (y_proba_te >= best_thr).astype(int), y_proba_te))

    print(f"Stage-3 wall time: {(time.time() - t0) / 60:.1f} min")

    def summarize(runs):
        keys = runs[0].keys()
        return {k: (np.mean([r[k] for r in runs]), np.std([r[k] for r in runs])) for k in keys}

    return summarize(default_runs), summarize(tuned_runs), np.mean(thresholds)


def evaluate_official_split(model, X_train, X_test, y_train, y_test):
    X_fit, X_thr, y_fit, y_thr = train_test_split(
        X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE
    )
    model.fit(X_fit, y_fit)

    y_proba_thr = model.predict_proba(X_thr)[:, 1]
    best_thr, _ = optimize_threshold(y_thr, y_proba_thr)

    y_proba_te = model.predict_proba(X_test)[:, 1]
    default_result = _score(y_test, (y_proba_te >= 0.5).astype(int), y_proba_te)
    tuned_result = _score(y_test, (y_proba_te >= best_thr).astype(int), y_proba_te)

    model.fit(X_train, y_train)
    return default_result, tuned_result, best_thr, model


# ---------------------------------------------------------------------
# 6. COMPARISON TABLE (unchanged)
# ---------------------------------------------------------------------
REFERENCE_RESULTS = {
    "Hakke et al. (2025) - LR": {
        "Accuracy": 0.8869, "Precision": 0.85, "Recall": 0.99,
        "F1-Score": 0.91, "AUC": np.nan,
    },
    "Kasongo & Sun (2020) - LR (42 features, no selection)": {
        "Accuracy": 0.7959, "Precision": 0.7332, "Recall": 0.9894,
        "F1-Score": 0.8422, "AUC": np.nan,
    },
}


def build_comparison_table(cv_default, cv_tuned, split_default, split_tuned,
                            out_csv=os.path.join(OUT_DIR, "comparison_table_logreg.csv")):
    rows = []
    for name, metrics in REFERENCE_RESULTS.items():
        row = {"Model": name}
        row.update(metrics)
        rows.append(row)

    def add_row(name, metrics_dict, is_cv):
        row = {"Model": name}
        for k, v in metrics_dict.items():
            row[k] = round(v[0], 4) if is_cv else round(v, 4)
        rows.append(row)

    add_row("Proposed LR -- 5-fold CV, default thr=0.50", cv_default, is_cv=True)
    add_row("Proposed LR -- 5-fold CV, tuned threshold", cv_tuned, is_cv=True)
    add_row("Proposed LR -- official split, default thr=0.50", split_default, is_cv=False)
    add_row("Proposed LR -- official split, tuned threshold", split_tuned, is_cv=False)

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(df.to_string(index=False))
    return df


# ---------------------------------------------------------------------
# 7. GRAPHS (unchanged)
# ---------------------------------------------------------------------
def plot_comparison_bar(df, out_path=os.path.join(OUT_DIR, "comparison_bar_logreg.png")):
    metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]
    x = np.arange(len(df))
    width = 0.2
    fig, ax = plt.subplots(figsize=(13, 6))
    for i, m in enumerate(metrics):
        ax.bar(x + i * width, df[m], width, label=m)
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(df["Model"], rotation=20, ha="right", fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title("Logistic Regression -- Proposed Model vs. Reference Papers")
    ax.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_confusion_matrix(model, X_test, y_test, threshold,
                           out_path=os.path.join(OUT_DIR, "confusion_matrix_logreg.png")):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Malicious"])
    fig, ax = plt.subplots(figsize=(5, 5))
    disp.plot(ax=ax, cmap="Blues", values_format="d")
    ax.set_title(f"Confusion Matrix -- Tuned LR (threshold={threshold:.2f})")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_roc(model, X_test, y_test, out_path=os.path.join(OUT_DIR, "roc_curve_logreg.png")):
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, color="steelblue", label=f"Tuned LR (AUC = {auc:.3f})")
    plt.plot([0, 1], [0, 1], "--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve -- Tuned Logistic Regression (All Features)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_threshold_curve(model, X_thr, y_thr, best_thr,
                          out_path=os.path.join(OUT_DIR, "threshold_curve_logreg.png")):
    y_proba = model.predict_proba(X_thr)[:, 1]
    thresholds = np.arange(0.01, 1.00, 0.01)
    f1s = [f1_score(y_thr, (y_proba >= t).astype(int), zero_division=0) for t in thresholds]
    plt.figure(figsize=(7, 5))
    plt.plot(thresholds, f1s, color="darkorange")
    plt.axvline(best_thr, color="gray", linestyle="--", label=f"Chosen threshold = {best_thr:.2f}")
    plt.axvline(0.5, color="lightgray", linestyle=":", label="Default threshold = 0.50")
    plt.xlabel("Decision Threshold")
    plt.ylabel("F1-Score (Attack class)")
    plt.title("Threshold Tuning Curve (on validation carve-out, not test data)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_coefficients(model, feature_names, out_path=os.path.join(OUT_DIR, "coefficients_logreg.png")):
    coefs = model.coef_[0]
    order = np.argsort(np.abs(coefs))[::-1]
    plt.figure(figsize=(9, max(6, 0.28 * len(order))))
    colors = ["crimson" if coefs[i] < 0 else "steelblue" for i in order]
    plt.barh([feature_names[i] for i in order][::-1], coefs[order][::-1], color=colors[::-1])
    plt.xlabel("Coefficient (on MinMax-scaled features -- directly comparable in magnitude)")
    plt.title("Logistic Regression Coefficients -- blue pushes toward 'attack', red toward 'normal'")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


# ---------------------------------------------------------------------
# 8. MAIN
# ---------------------------------------------------------------------
if __name__ == "__main__":
    TRAIN_CSV = "UNSW_NB15_training-set.csv"
    TEST_CSV = "UNSW_NB15_testing-set.csv"

    pipeline_start = time.time()

    print("Loading data...")
    df_train, df_test = load_unsw_nb15(train_path=TRAIN_CSV, test_path=TEST_CSV)

    print("Preprocessing (no feature selection -- all columns kept, "
          "encoders/scaler fit on train only)...")
    X_train, X_test, y_train, y_test, scaler, encoders = preprocess_train_test(df_train, df_test)

    X_full = pd.concat([X_train, X_test], axis=0).reset_index(drop=True)
    y_full = pd.concat([y_train, y_test], axis=0).reset_index(drop=True)

    print("\nHyperparameter tuning (Random -> Grid, penalty-appropriate solver, capped search max_iter)...")
    t0 = time.time()
    best_model, best_params = tune_logreg(X_train, y_train)
    print(f"\nTotal tuning time: {(time.time() - t0) / 60:.1f} min")

    print("\n5-fold CV evaluation (default vs. tuned threshold) on the FULL dataset...")
    cv_default, cv_tuned, mean_thr = evaluate_cv(best_model, X_full, y_full, cv_splits=5)
    print(f"Mean tuned threshold across folds: {mean_thr:.3f}")
    for label, results in [("Default (0.50)", cv_default), ("Tuned", cv_tuned)]:
        print(f"\n-- CV, {label} threshold --")
        for k, (m, s) in results.items():
            print(f"{k}: {m:.4f} +/- {s:.4f}")

    print("\nEvaluating on the OFFICIAL train/test split (matches the papers' own protocol)...")
    split_default, split_tuned, best_thr, fitted_model = evaluate_official_split(
        best_model, X_train, X_test, y_train, y_test
    )
    print(f"Threshold tuned on official split: {best_thr:.3f}")
    for label, results in [("Default (0.50)", split_default), ("Tuned", split_tuned)]:
        print(f"\n-- Official split, {label} threshold --")
        for k, v in results.items():
            print(f"{k}: {v:.4f}")

    print("\nBuilding comparison table against the 2 reference papers...")
    comp_df = build_comparison_table(cv_default, cv_tuned, split_default, split_tuned)

    print("\nGenerating graphs...")
    X_fit, X_thr_plot, y_fit, y_thr_plot = train_test_split(
        X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE
    )
    plot_comparison_bar(comp_df)
    plot_confusion_matrix(fitted_model, X_test, y_test, best_thr)
    plot_roc(fitted_model, X_test, y_test)
    plot_threshold_curve(fitted_model, X_thr_plot, y_thr_plot, best_thr)
    plot_coefficients(fitted_model, X_full.columns.tolist())

    print(f"\nAll outputs saved to ./{OUT_DIR}/ (with *_logreg suffix)")
    total_min = (time.time() - pipeline_start) / 60
    print(f"\nTOTAL PIPELINE TIME: {total_min:.1f} min "
          f"({'within' if total_min <= 120 else 'OVER'} the 1-2 hour target)")

Loading data...
Preprocessing (no feature selection -- all columns kept, encoders/scaler fit on train only)...
Training features : (175341, 42)
Testing features  : (82332, 42)

Hyperparameter tuning (Random -> Grid, penalty-appropriate solver, capped search max_iter)...


Stage 1: Random search:   0%|          | 0/200 [00:00<?, ?it/s]

[Stage 1: Random search] Time budget (20 min) reached -- stopped after 14/40 candidates. Using the best one found so far.
[Stage 1: Random search] Best: {'penalty': 'l1', 'C': 1.0, 'class_weight': None} (solver=liblinear)  CV F1 = 0.9526
Stage-1 wall time: 32.5 min


Stage 2: Grid refine:   0%|          | 0/15 [00:00<?, ?it/s]

[Stage 2: Grid refine] Best: {'penalty': 'l1', 'C': 3.0, 'class_weight': None} (solver=liblinear)  CV F1 = 0.9528
Stage-2 wall time: 17.7 min

Final tuned hyperparameters: {'penalty': 'l1', 'C': 3.0, 'class_weight': None}  (solver=liblinear, max_iter=5000)

Total tuning time: 50.3 min

5-fold CV evaluation (default vs. tuned threshold) on the FULL dataset...


Stage 3: Final 5-fold CV:   0%|          | 0/5 [00:00<?, ?it/s]

Stage-3 wall time: 13.6 min
Mean tuned threshold across folds: 0.496

-- CV, Default (0.50) threshold --
Accuracy: 0.8979 +/- 0.0012
Precision: 0.8847 +/- 0.0011
Recall: 0.9661 +/- 0.0018
F1-Score: 0.9236 +/- 0.0010
AUC: 0.9659 +/- 0.0004

-- CV, Tuned threshold --
Accuracy: 0.8977 +/- 0.0012
Precision: 0.8858 +/- 0.0101
Recall: 0.9646 +/- 0.0126
F1-Score: 0.9234 +/- 0.0007
AUC: 0.9659 +/- 0.0004

Evaluating on the OFFICIAL train/test split (matches the papers' own protocol)...
Threshold tuned on official split: 0.440

-- Official split, Default (0.50) threshold --
Accuracy: 0.8026
Precision: 0.7484
Recall: 0.9664
F1-Score: 0.8435
AUC: 0.9325

-- Official split, Tuned threshold --
Accuracy: 0.8021
Precision: 0.7445
Recall: 0.9752
F1-Score: 0.8444
AUC: 0.9325

Building comparison table against the 2 reference papers...
                                                Model  Accuracy  Precision  Recall  F1-Score    AUC
                             Hakke et al. (2025) - LR    0.8869     0.